# LLM-EEG Framework v2: RL-Based Adaptive Preprocessing & Decision Validation for MI-BCI

---

**Author:** [Your Name] | **Institution:** [Your University] | **Date:** February 2025 | **Version:** 2.0
**Runtime:** Google Colab Pro (A100 GPU recommended) | **Repository:** https://github.com/erlika/llm-eeg

---

## Abstract

This notebook presents the **LLM-EEG Framework**, a novel three-pillar architecture for Motor Imagery Brain-Computer Interfaces (MI-BCI):

1. **Adaptive Preprocessing Agent (APA)** -- RL (Q-learning) agent that dynamically selects preprocessing profiles based on signal quality.
2. **Decision Validation Agent (DVA)** -- Multi-validator architecture for Accept/Reject/Review decisions on classifications.
3. **LLM Explainability** -- Human-readable explanations of agent decisions.

Evaluated on **BCI Competition IV-2a** (9 subjects, 4-class MI, 22 EEG, 250 Hz) with 10 classifiers: LDA, SVM, EEGNet, ShallowConvNet, DeepConvNet, ATCNet, EEGConformer, EEGTCNet, CTNet, MSCFormer. Supports both session-wise (T/E) and k-fold CV evaluation protocols.

**Keywords:** Motor Imagery, BCI, Reinforcement Learning, Adaptive Preprocessing, Decision Validation, EEG, Deep Learning, Q-Learning, XAI

## Table of Contents

| # | Section |
|---|---------|
| 1 | [Introduction & Contributions](#sec1) |
| 2 | [Environment Setup](#sec2) |
| 3 | [Framework Definitions](#sec3) |
| 4 | [Data Exploration](#sec4) |
| 5 | [Run Full Experiment](#sec5) |
| 6 | [Results & Visualization](#sec6) |
| 7 | [Statistical Tests](#sec7) |
| 8 | [Literature Comparison & Differentiation](#sec8) |
| 9 | [Discussion & Conclusions](#sec9) |
| 10 | [Export & References](#sec10) |

## 1. Introduction <a name="sec1"></a>

Motor Imagery BCIs decode imagined movements from EEG. Despite advances in deep learning (AMEEGNet, CIACNet, CLTNet, MSCFormer, BrainGridNet, EEGEncoder -- 2024-2025), three gaps persist:

1. **Static preprocessing** -- Fixed filtering regardless of per-trial signal quality.
2. **No decision validation** -- Classifier outputs trusted blindly.
3. **Black-box models** -- No interpretable explanations.

### Contributions (Novelty Triangle)

| Component | Innovation | Gap |
|-----------|-----------|-----|
| **APA** | RL per-trial preprocessing (64 states, 3 actions) | Static pipelines |
| **DVA** | Weighted multi-validator decisions | No validation |
| **LLM** | Natural language explanations | Black-box |
| **Cross-trial** | Q-table transfers across sessions | No adaptation |
| **Modular** | Plugin interfaces | Monolithic designs |

### APA: 64 states (4 SNR x 4 artifact x 4 noise), 3 actions (Conservative/Moderate/Aggressive), Q-learning (alpha=0.1, gamma=0.99, epsilon 1.0->0.01)
### DVA: Confidence(w=0.4) + Margin(w=0.3) + Quality(w=0.3), Accept>=0.8, Reject<0.5

## 2. Environment Setup <a name="sec2"></a>

In [ ]:
# ==============================================================================
# Environment Setup
# ==============================================================================
import os, sys, subprocess, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
print(f"Running on Google Colab: {IN_COLAB}")

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'seaborn',
    'pandas', 'tqdm', 'pyyaml', 'braindecode', 'einops'], capture_output=True)

try:
    import torch
    print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], capture_output=True)
    import torch
    print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

REPO_URL = 'https://github.com/erlika/llm-eeg.git'
REPO_DIR = '/content/llm-eeg' if IN_COLAB else os.getcwd()
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR} 2>/dev/null || echo "Clone skipped"')
    if os.path.exists(REPO_DIR):
        sys.path.insert(0, REPO_DIR); os.chdir(REPO_DIR)
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DIR = '/content/drive/MyDrive/LLM-EEG'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DATA_DIR = os.path.join(DRIVE_DIR, 'data')
    RESULTS_DIR = os.path.join(DRIVE_DIR, 'results')
else:
    sys.path.insert(0, os.getcwd())
    DATA_DIR, RESULTS_DIR = 'data', 'results'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---- Data Availability Check ----
USE_REAL_DATA = False
REAL_DATA_FILES = []
EVAL_DATA_FILES = []
for subj in range(1, 10):
    t_path = os.path.join(DATA_DIR, f'A0{subj}T.mat')
    e_path = os.path.join(DATA_DIR, f'A0{subj}E.mat')
    if os.path.exists(t_path):
        REAL_DATA_FILES.append(t_path)
    if os.path.exists(e_path):
        EVAL_DATA_FILES.append(e_path)

if REAL_DATA_FILES:
    USE_REAL_DATA = True
    print(f"\n*** REAL BCI IV-2a DATA FOUND ***")
    print(f"  Data directory: {DATA_DIR}")
    print(f"  Training files: {len(REAL_DATA_FILES)} (.mat)")
    print(f"  Evaluation files: {len(EVAL_DATA_FILES)} (.mat)")
    for f in REAL_DATA_FILES:
        sz = os.path.getsize(f) / 1e6
        print(f"    {os.path.basename(f)} ({sz:.1f} MB)")
    for f in EVAL_DATA_FILES:
        sz = os.path.getsize(f) / 1e6
        print(f"    {os.path.basename(f)} ({sz:.1f} MB)")
    if EVAL_DATA_FILES:
        print(f"  Mode: SESSION-WISE evaluation (train on T, test on E)")
    else:
        print(f"  Mode: k-fold CV on training session (no E files found)")
        print(f"  To enable session-wise eval, download A01E.mat-A09E.mat")
else:
    print(f"\n*** No real BCI IV-2a data found in: {DATA_DIR} ***")
    print(f"  Expected files: A01T.mat .. A09T.mat")
    if IN_COLAB:
        print(f"  Please ensure your data is in Google Drive: MyDrive/LLM-EEG/data/")
    print(f"  Mode: SYNTHETIC DATA (fallback) -- results are for demonstration only")

print(f"\nData: {DATA_DIR}")
print(f"Results: {RESULTS_DIR}")
print(f"Use Real Data: {USE_REAL_DATA}")


In [ ]:
# ==============================================================================
# Import Libraries & Configure Plotting
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal
from scipy import stats as scipy_stats
from scipy.linalg import eigh
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,
                             classification_report, precision_score, recall_score, f1_score)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import json, time, copy

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams.update({
    'figure.figsize': (12, 6), 'figure.dpi': 150, 'savefig.dpi': 300,
    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 9,
    'figure.titlesize': 15, 'lines.linewidth': 1.8,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.1,
})
COLORS = {'baseline': '#4C72B0', 'apa': '#DD8452', 'dva': '#55A868',
           'conservative': '#55A868', 'moderate': '#4C72B0', 'aggressive': '#C44E52'}

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} | NumPy {np.__version__} | PyTorch {torch.__version__}")


## 3. Core Framework Definitions <a name="sec3"></a>

All components defined inline for Colab self-containedness.

In [ ]:
# ==============================================================================
# Framework Definitions (loaded from run_experiment.py)
# All classes: SyntheticBCI2aGenerator, EEGPreprocessor, CSPFeatureExtractor,
# BandPowerExtractor, ClassifierFactory, _TorchClassifierWrapper,
# APAAgentLite, DVAAgentLite, EvaluationMetrics, ExperimentRunner
# ==============================================================================
import os, sys, json, time, logging, warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Any, Tuple

import numpy as np
from scipy import signal as scipy_signal
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('.') if not 'IN_COLAB' in dir() else Path('.')
sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.WARNING, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
logger = logging.getLogger('LLM-EEG')

EXPERIMENT_CONFIG = {
    'dataset': {
        'name': 'BCI Competition IV-2a',
        'n_subjects': 9,
        'n_sessions': 2,
        'n_classes': 4,
        'class_names': ['Left Hand', 'Right Hand', 'Feet', 'Tongue'],
        'n_channels': 22,
        'sampling_rate': 250,
        'trial_duration': 3.5,  # seconds (MI window: 2.5-6.0s = 875 samples)
        'n_trials_per_class': 72,
        'total_trials_per_session': 288,
    },
    'preprocessing': {
        'notch_freq': 50,
        'bandpass_low': 8,
        'bandpass_high': 30,
        'filter_order': 5,
    },
    'features': {
        'csp_components': 6,
        'csp_reg': 0.01,
        'band_power_bands': {
            'mu': (8, 12),
            'beta_low': (12, 20),
            'beta_high': (20, 30),
        },
    },
    'classifiers': {
        'lda': {'solver': 'svd', 'shrinkage': None},
        'svm': {'kernel': 'rbf', 'C': 1.0, 'gamma': 'scale'},
        'eegnet': {
            'F1': 8, 'D': 2, 'F2': 16,
            'dropout_rate': 0.5, 'epochs': 200,
            'batch_size': 64, 'learning_rate': 0.001,
            'patience': 30, 'weight_decay': 0,
            'grad_clip': 5.0,
        },
        'shallow_convnet': {
            'epochs': 200, 'batch_size': 64,
            'learning_rate': 6.25e-4, 'patience': 30,
            'weight_decay': 0, 'grad_clip': 5.0,
        },
        'deep_convnet': {
            'epochs': 200, 'batch_size': 64,
            'learning_rate': 1e-3, 'patience': 30,
            'weight_decay': 5e-4, 'grad_clip': 5.0,
        },
        'atcnet': {
            'epochs': 200, 'batch_size': 64,
            'learning_rate': 1e-3, 'patience': 30,
            'weight_decay': 0, 'grad_clip': 5.0,
        },
        'eegconformer': {
            'epochs': 200, 'batch_size': 72,
            'learning_rate': 5e-4, 'patience': 30,
            'weight_decay': 1e-2, 'grad_clip': None,
        },
        'eegtcnet': {
            'epochs': 200, 'batch_size': 64,
            'learning_rate': 1e-3, 'patience': 30,
            'weight_decay': 1e-3, 'grad_clip': 5.0,
        },
        'ctnet': {
            'epochs': 200, 'batch_size': 64,
            'learning_rate': 5e-4, 'patience': 30,
            'weight_decay': 1e-3, 'grad_clip': 5.0,
        },
        'mscformer': {
            'epochs': 200, 'batch_size': 128,
            'learning_rate': 1e-3, 'patience': 30,
            'dropout_rate': 0.5, 'pooling_size': 44,
            'weight_decay': 1e-2, 'grad_clip': None,
        },
    },
    'apa': {
        'policy_type': 'q_learning',
        'learning_rate': 0.1,
        'discount_factor': 0.99,
        'epsilon_start': 1.0,
        'epsilon_decay': 0.995,
        'epsilon_min': 0.01,
        'deep_model_enabled': False,
        'state_bins': {
            'snr': [0, 5, 10, 20, float('inf')],
            'artifact_ratio': [0, 0.1, 0.3, 0.5, 1.0],
            'line_noise': [0, 0.5, 1.0, 2.0, float('inf')],
        },
    },
    'dva': {
        'accept_threshold': 0.8,
        'reject_threshold': 0.5,
        'review_threshold': 0.65,
    },
    'training': {
        'use_ema': True,
    },
    'evaluation': {
        'random_seed': 42,
        'n_folds': 5,
        'test_size': 0.2,
    },
}

# ============================================================================
# SYNTHETIC DATA GENERATOR
# ============================================================================

class SyntheticBCI2aGenerator:
    """
    Generates realistic synthetic BCI Competition IV-2a data.
    
    Simulates motor imagery EEG with:
    - ERD/ERS patterns in mu (8-12 Hz) and beta (12-30 Hz) bands
    - Subject-specific SNR variations
    - 1/f noise background
    - Realistic class-discriminative patterns at C3/C4
    """
    
    def __init__(self, config: dict, random_seed: int = 42):
        self.config = config
        self.rng = np.random.RandomState(random_seed)
        self.n_channels = config['dataset']['n_channels']
        self.fs = config['dataset']['sampling_rate']
        self.trial_duration = config['dataset']['trial_duration']
        self.n_samples = int(self.fs * self.trial_duration)  # 875 samples for 3.5s window
        self.n_classes = config['dataset']['n_classes']
        self.class_names = config['dataset']['class_names']
        
        # BCI IV-2a channel names (22 EEG channels)
        self.channel_names = [
            'Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
            'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6',
            'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
            'P1', 'Pz', 'P2', 'POz'
        ]
        
        # Motor cortex channel indices (C3=7, C4=11, C1=8, C2=10)
        self.motor_channels = {
            'C3': 7, 'C4': 11, 'C1': 8, 'C2': 10,
            'FC3': 1, 'FC4': 5, 'CP3': 13, 'CP4': 17
        }
    
    def generate_subject(self, subject_id: int, 
                        n_trials_per_class: int = 72) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate synthetic data for one subject.
        
        Args:
            subject_id: Subject identifier (1-9)
            n_trials_per_class: Trials per class
            
        Returns:
            X: (n_trials, n_channels, n_samples)
            y: (n_trials,) with labels 0-3
        """
        # Subject-specific parameters (simulate inter-subject variability)
        subject_snr = 5.0 + subject_id * 1.5 + self.rng.randn() * 2.0
        subject_alpha_power = 0.5 + self.rng.rand() * 0.5
        
        all_trials = []
        all_labels = []
        
        for class_idx in range(self.n_classes):
            for trial_idx in range(n_trials_per_class):
                trial = self._generate_trial(
                    class_idx, subject_snr, subject_alpha_power
                )
                all_trials.append(trial)
                all_labels.append(class_idx)
        
        X = np.array(all_trials)
        y = np.array(all_labels)
        
        # Shuffle
        shuffle_idx = self.rng.permutation(len(y))
        X = X[shuffle_idx]
        y = y[shuffle_idx]
        
        return X, y
    
    def _generate_trial(self, class_idx: int, snr: float, 
                       alpha_power: float) -> np.ndarray:
        """Generate a single trial with class-specific patterns."""
        t = np.arange(self.n_samples) / self.fs
        trial = np.zeros((self.n_channels, self.n_samples))
        
        # 1. Background 1/f noise for all channels
        for ch in range(self.n_channels):
            trial[ch] = self._generate_pink_noise(self.n_samples) * 10
        
        # 2. Add alpha rhythm (8-13 Hz) to posterior channels
        alpha_freq = 10 + self.rng.randn() * 0.5
        alpha_signal = alpha_power * np.sin(2 * np.pi * alpha_freq * t + self.rng.rand() * 2 * np.pi)
        for ch in [18, 19, 20, 21]:  # P1, Pz, P2, POz
            trial[ch] += alpha_signal * (5 + self.rng.randn())
        
        # 3. Class-specific motor imagery patterns
        # ERD (Event-Related Desynchronization) in mu/beta bands
        mu_freq = 10 + self.rng.randn() * 0.3
        beta_freq = 20 + self.rng.randn() * 2
        
        erd_envelope = self._erd_envelope(t)
        
        if class_idx == 0:  # Left Hand → ERD at C4 (contralateral)
            self._add_erd(trial, 'C4', mu_freq, beta_freq, erd_envelope, snr)
            self._add_ers(trial, 'C3', mu_freq, erd_envelope, snr * 0.3)
        elif class_idx == 1:  # Right Hand → ERD at C3
            self._add_erd(trial, 'C3', mu_freq, beta_freq, erd_envelope, snr)
            self._add_ers(trial, 'C4', mu_freq, erd_envelope, snr * 0.3)
        elif class_idx == 2:  # Feet → ERD at Cz
            self._add_erd_channel(trial, 9, mu_freq, beta_freq, erd_envelope, snr)  # Cz
            self._add_erd_channel(trial, 15, mu_freq, beta_freq, erd_envelope, snr * 0.5)  # CPz
        elif class_idx == 3:  # Tongue → bilateral ERD
            self._add_erd(trial, 'C3', mu_freq, beta_freq, erd_envelope, snr * 0.6)
            self._add_erd(trial, 'C4', mu_freq, beta_freq, erd_envelope, snr * 0.6)
        
        # 4. Add 50 Hz line noise
        line_noise = 0.5 * np.sin(2 * np.pi * 50 * t)
        trial += line_noise[np.newaxis, :] * (0.5 + self.rng.rand(self.n_channels, 1) * 0.5)
        
        # 5. Add measurement noise
        noise_level = np.max(np.abs(trial)) / (10 ** (snr / 20))
        trial += self.rng.randn(self.n_channels, self.n_samples) * noise_level
        
        return trial
    
    def _generate_pink_noise(self, n_samples: int) -> np.ndarray:
        """Generate 1/f noise."""
        white = self.rng.randn(n_samples)
        fft = np.fft.rfft(white)
        freqs = np.fft.rfftfreq(n_samples, 1.0 / self.fs)
        freqs[0] = 1  # Avoid division by zero
        fft = fft / np.sqrt(freqs)
        return np.fft.irfft(fft, n_samples)
    
    def _erd_envelope(self, t: np.ndarray) -> np.ndarray:
        """Create ERD temporal envelope (onset at 0.5s, peak at 2s)."""
        envelope = np.zeros_like(t)
        mask = t >= 0.5
        t_shifted = t[mask] - 0.5
        envelope[mask] = 1.0 - np.exp(-t_shifted / 0.5)
        # Ramp down after 3s
        mask2 = t >= 3.0
        t_shifted2 = t[mask2] - 3.0
        envelope[mask2] *= np.exp(-t_shifted2 / 0.5)
        return envelope
    
    def _add_erd(self, trial: np.ndarray, region: str, mu_freq: float,
                beta_freq: float, envelope: np.ndarray, snr: float):
        """Add ERD pattern to a motor region (C3 or C4 and neighbors)."""
        t = np.arange(self.n_samples) / self.fs
        
        if region == 'C3':
            channels = [self.motor_channels[k] for k in ['C3', 'C1', 'FC3', 'CP3'] 
                       if k in self.motor_channels]
        else:
            channels = [self.motor_channels[k] for k in ['C4', 'C2', 'FC4', 'CP4']
                       if k in self.motor_channels]
        
        for ch in channels:
            weight = 1.0 if ch in [self.motor_channels.get('C3'), self.motor_channels.get('C4')] else 0.6
            self._add_erd_channel(trial, ch, mu_freq, beta_freq, envelope, snr * weight)
    
    def _add_erd_channel(self, trial: np.ndarray, ch: int, mu_freq: float,
                        beta_freq: float, envelope: np.ndarray, snr: float):
        """Add ERD to a specific channel."""
        t = np.arange(self.n_samples) / self.fs
        phase = self.rng.rand() * 2 * np.pi
        
        # Mu band ERD (power decrease)
        mu_signal = np.sin(2 * np.pi * mu_freq * t + phase) * (1.0 - 0.7 * envelope)
        trial[ch] += mu_signal * snr * 0.5
        
        # Beta band ERD
        beta_signal = np.sin(2 * np.pi * beta_freq * t + phase * 2) * (1.0 - 0.5 * envelope)
        trial[ch] += beta_signal * snr * 0.3
    
    def _add_ers(self, trial: np.ndarray, region: str, mu_freq: float,
                envelope: np.ndarray, snr: float):
        """Add ERS (synchronization) pattern."""
        t = np.arange(self.n_samples) / self.fs
        
        if region == 'C3':
            ch = self.motor_channels['C3']
        else:
            ch = self.motor_channels['C4']
        
        phase = self.rng.rand() * 2 * np.pi
        ers_signal = np.sin(2 * np.pi * mu_freq * t + phase) * (1.0 + 0.3 * envelope)
        trial[ch] += ers_signal * snr * 0.3
    
    def get_dataset_info(self) -> dict:
        """Return dataset metadata."""
        return {
            'name': self.config['dataset']['name'],
            'n_channels': self.n_channels,
            'sampling_rate': self.fs,
            'channel_names': self.channel_names,
            'n_classes': self.n_classes,
            'class_names': self.class_names,
            'trial_duration': self.trial_duration,
            'n_samples': self.n_samples,
        }



# ============================================================================
# REAL BCI IV-2a DATA LOADER (.mat format)
# ============================================================================

class RealBCI2aLoader:
    """
    Loads real BCI Competition IV-2a data from .mat files.
    
    Supports the standard .mat format available on Kaggle / BNCI Horizon:
      - Files: A01T.mat .. A09T.mat (training), A01E.mat .. A09E.mat (evaluation)
      - Structure: mat['data'] -> array of runs, each run contains:
        [X (samples x channels), trial_onsets, y (labels), fs, classes, artifacts, gender, age]
      - 22 EEG channels + 3 EOG channels (only EEG used)
      - 250 Hz sampling rate
      - 4 MI classes: Left Hand (1), Right Hand (2), Feet (3), Tongue (4)
    
    Reference: MultiScale-BCI/IV-2a (github.com/MultiScale-BCI/IV-2a)
    """
    
    def __init__(self, data_dir: str, config: dict = None):
        self.data_dir = Path(data_dir)
        self.config = config or EXPERIMENT_CONFIG
        self.n_channels = 22  # EEG only (exclude 3 EOG)
        self.fs = 250
        self.trial_window = 7 * self.fs  # 7s window from trial onset (0-7s)
        self.mi_start = int(2.5 * self.fs)  # MI onset at t=2.5s (post-cue stabilization)
        self.mi_end = int(6.0 * self.fs)    # MI ends at t=6s
        self.mi_samples = self.mi_end - self.mi_start  # 3.5s = 875 samples
        self.n_classes = 4
        self.class_names = ['Left Hand', 'Right Hand', 'Feet', 'Tongue']
        self.channel_names = [
            'Fz', 'FC3', 'FC1', 'FCz', 'FC2', 'FC4',
            'C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6',
            'CP3', 'CP1', 'CPz', 'CP2', 'CP4',
            'P1', 'Pz', 'P2', 'POz'
        ]
    
    def is_available(self) -> bool:
        """Check if real data files exist."""
        if not self.data_dir.exists():
            return False
        # Check for at least one training file
        for subj in range(1, 10):
            if (self.data_dir / f'A0{subj}T.mat').exists():
                return True
        return False
    
    def get_available_subjects(self) -> list:
        """Return list of available subject IDs."""
        subjects = []
        for subj in range(1, 10):
            if (self.data_dir / f'A0{subj}T.mat').exists():
                subjects.append(subj)
        return subjects
    
    def load_subject(self, subject_id: int, 
                     training: bool = True,
                     mi_period_only: bool = True) -> tuple:
        """
        Load data for one subject.
        
        Args:
            subject_id: Subject number (1-9)
            training: If True, load training (T) file; else evaluation (E)
            mi_period_only: If True, extract only the 4s MI period (t=2-6s);
                          If False, return 7s window from trial onset
        
        Returns:
            X: (n_valid_trials, 22, n_samples) - EEG data
            y: (n_valid_trials,) - class labels (0-indexed: 0=LH, 1=RH, 2=Feet, 3=Tongue)
        """
        import scipy.io as sio
        
        suffix = 'T' if training else 'E'
        filepath = self.data_dir / f'A0{subject_id}{suffix}.mat'
        
        if not filepath.exists():
            raise FileNotFoundError(f"Data file not found: {filepath}")
        
        logger.info(f"Loading {filepath.name} ({filepath.stat().st_size / 1e6:.1f} MB)...")
        
        mat = sio.loadmat(str(filepath), struct_as_record=False, squeeze_me=True)
        
        # Determine output sample length
        if mi_period_only:
            n_samples_out = self.mi_samples  # 1000 samples (4s)
        else:
            n_samples_out = self.trial_window  # 1750 samples (7s)
        
        # Pre-allocate for max possible trials (288 per session)
        max_trials = 288
        data_return = np.zeros((max_trials, self.n_channels, n_samples_out))
        class_return = np.zeros(max_trials)
        n_valid = 0
        
        a_data = mat['data']
        
        # Handle both squeezed and unsqueezed formats
        if a_data.ndim == 0:
            # Single element - wrap it
            runs = [a_data.item()]
        elif a_data.ndim == 1:
            runs = a_data
        else:
            # 2D array (1, n_runs)
            runs = [a_data[0, ii] for ii in range(a_data.shape[1])]
        
        for run_data in runs:
            try:
                # Extract fields from the run structure
                if hasattr(run_data, '_fieldnames'):
                    # Structured object
                    X_cont = np.array(run_data.X, dtype=np.float64)
                    trial_onsets = np.array(run_data.trial, dtype=np.int64).flatten()
                    y_labels = np.array(run_data.y, dtype=np.int64).flatten()
                    artifacts = np.array(run_data.artifacts, dtype=np.int64).flatten()
                else:
                    # Nested array format: [X, trial, y, fs, classes, artifacts, ...]
                    fields = run_data[0, 0] if run_data.ndim >= 2 else run_data.item()
                    X_cont = np.array(fields[0], dtype=np.float64)
                    trial_onsets = np.array(fields[1], dtype=np.int64).flatten()
                    y_labels = np.array(fields[2], dtype=np.int64).flatten()
                    artifacts = np.array(fields[5], dtype=np.int64).flatten()
                
                if len(trial_onsets) == 0 or len(y_labels) == 0:
                    continue  # Skip runs with no trials (e.g., EOG calibration)
                
                for trial_idx in range(len(trial_onsets)):
                    # Skip artifact trials
                    if trial_idx < len(artifacts) and artifacts[trial_idx] != 0:
                        continue
                    
                    onset = int(trial_onsets[trial_idx])
                    
                    if mi_period_only:
                        # Extract 4s MI period: t=2s to t=6s relative to trial onset
                        start = onset + self.mi_start
                        end = onset + self.mi_end
                    else:
                        # Extract full 7s window
                        start = onset
                        end = onset + self.trial_window
                    
                    # Bounds check
                    if end > X_cont.shape[0]:
                        continue
                    
                    # Extract and transpose: (samples, channels) -> (channels, samples)
                    segment = X_cont[start:end, :self.n_channels].T
                    
                    if segment.shape[1] == n_samples_out:
                        data_return[n_valid] = segment
                        # Convert to 0-indexed labels (original: 1-4 -> 0-3)
                        class_return[n_valid] = int(y_labels[trial_idx]) - 1
                        n_valid += 1
                        
            except Exception as e:
                logger.warning(f"  Skipping run: {e}")
                continue
        
        X = data_return[:n_valid]
        y = class_return[:n_valid].astype(int)
        
        logger.info(f"  Loaded {n_valid} valid trials, shape: X={X.shape}, y={y.shape}")
        logger.info(f"  Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
        
        return X, y
    
    def get_available_eval_subjects(self) -> List[int]:
        """Return list of subject IDs that have evaluation (E) files."""
        subjects = []
        for subj in range(1, 10):
            if (self.data_dir / f'A0{subj}E.mat').exists():
                subjects.append(subj)
        return subjects
    
    def load_both_sessions(self, subject_id: int,
                           mi_period_only: bool = True) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
        """Load both training (T) and evaluation (E) sessions.
        
        Returns:
            X_train, y_train: Training session data and labels
            X_test, y_test: Evaluation session data and labels (None if E file missing)
        """
        X_train, y_train = self.load_subject(subject_id, training=True, mi_period_only=mi_period_only)
        
        e_path = self.data_dir / f'A0{subject_id}E.mat'
        if e_path.exists():
            X_test, y_test = self.load_subject(subject_id, training=False, mi_period_only=mi_period_only)
            logger.info(f"  Session-wise: Train=({X_train.shape}), Test=({X_test.shape})")
            return X_train, y_train, X_test, y_test
        else:
            logger.info(f"  No evaluation file for subject {subject_id}, will use CV")
            return X_train, y_train, None, None

    def get_dataset_info(self) -> dict:
        """Return dataset metadata."""
        return {
            'name': 'BCI Competition IV-2a (Real Data)',
            'n_channels': self.n_channels,
            'sampling_rate': self.fs,
            'channel_names': self.channel_names,
            'n_classes': self.n_classes,
            'class_names': self.class_names,
            'trial_duration': self.mi_samples / self.fs,
            'n_samples': self.mi_samples,
            'data_dir': str(self.data_dir),
            'available_subjects': self.get_available_subjects(),
            'available_eval_subjects': self.get_available_eval_subjects(),
        }


# ============================================================================
# PREPROCESSING MODULE
# ============================================================================

class EEGPreprocessor:
    """Standard EEG preprocessing pipeline."""
    
    def __init__(self, config: dict):
        self.fs = config['dataset']['sampling_rate']
        self.notch_freq = config['preprocessing']['notch_freq']
        self.bp_low = config['preprocessing']['bandpass_low']
        self.bp_high = config['preprocessing']['bandpass_high']
        self.bp_order = config['preprocessing']['filter_order']
    
    def process(self, X: np.ndarray, 
                profile: str = 'moderate') -> np.ndarray:
        """
        Apply preprocessing pipeline.
        
        Args:
            X: (n_trials, n_channels, n_samples) or (n_channels, n_samples)
            profile: 'conservative', 'moderate', 'aggressive'
            
        Returns:
            Preprocessed data (same shape)
        """
        profiles = {
            'conservative': {'bp': (4, 40), 'order': 4, 'norm': True},
            'moderate': {'bp': (8, 30), 'order': 5, 'norm': True},
            'aggressive': {'bp': (8, 25), 'order': 6, 'norm': True},
            'deep': {'bp': (4, 40), 'order': 2, 'norm': False},
        }
        p = profiles.get(profile, profiles['moderate'])
        
        if X.ndim == 2:
            return self._process_single(X, p)
        elif X.ndim == 3:
            result = np.zeros_like(X)
            for i in range(X.shape[0]):
                result[i] = self._process_single(X[i], p)
            return result
        return X
    
    def _process_single(self, data: np.ndarray, profile: dict) -> np.ndarray:
        """Process a single trial (channels, samples)."""
        processed = data.copy()
        
        # 1. Notch filter (50 Hz)
        b_notch, a_notch = scipy_signal.iirnotch(
            self.notch_freq, Q=30, fs=self.fs
        )
        for ch in range(processed.shape[0]):
            processed[ch] = scipy_signal.filtfilt(b_notch, a_notch, processed[ch])
        
        # 2. Bandpass filter
        low, high = profile['bp']
        nyq = self.fs / 2
        b_bp, a_bp = scipy_signal.butter(
            profile['order'],
            [low / nyq, high / nyq],
            btype='band'
        )
        for ch in range(processed.shape[0]):
            processed[ch] = scipy_signal.filtfilt(b_bp, a_bp, processed[ch])
        
        # 3. Z-score normalization per channel
        if profile.get('norm', True):
            for ch in range(processed.shape[0]):
                std = np.std(processed[ch])
                if std > 1e-10:
                    processed[ch] = (processed[ch] - np.mean(processed[ch])) / std
        
        return processed
    
    def compute_signal_quality(self, trial: np.ndarray) -> dict:
        """Compute signal quality metrics for APA."""
        if trial.ndim == 3:
            trial = trial[0]  # Take first trial
        
        # SNR estimation
        signal_power = np.mean(np.var(trial, axis=1))
        noise_est = np.mean(np.abs(np.diff(trial, axis=1)), axis=1)
        noise_power = np.mean(noise_est ** 2)
        snr = 10 * np.log10(signal_power / (noise_power + 1e-10))
        snr = np.clip(snr, 0, 30)
        
        # Artifact ratio
        threshold = 5 * np.std(trial)
        artifact_samples = np.sum(np.abs(trial) > threshold)
        artifact_ratio = artifact_samples / trial.size
        artifact_ratio = np.clip(artifact_ratio, 0, 1)
        
        # Line noise power (50 Hz)
        freqs, psd = scipy_signal.welch(
            trial[0], fs=self.fs, nperseg=min(256, trial.shape[1])
        )
        idx_50 = np.argmin(np.abs(freqs - 50))
        line_noise = psd[idx_50] / np.mean(psd) if np.mean(psd) > 0 else 0
        line_noise = np.clip(line_noise, 0, 5)
        
        return {
            'snr': float(snr),
            'artifact_ratio': float(artifact_ratio),
            'line_noise': float(line_noise),
            'signal_quality_score': float(
                0.4 * min(1, snr / 20) + 
                0.4 * (1 - artifact_ratio) + 
                0.2 * (1 - min(1, line_noise / 2))
            ),
        }


# ============================================================================
# FEATURE EXTRACTION MODULE
# ============================================================================

class CSPFeatureExtractor:
    """Common Spatial Patterns feature extractor (multi-class OVR)."""
    
    def __init__(self, n_components: int = 6, reg: float = 0.01):
        self.n_components = n_components
        self.reg = reg
        self.filters_ = None
        self.classes_ = None
    
    def fit(self, X: np.ndarray, y: np.ndarray):
        """Fit CSP filters using One-vs-Rest for multi-class."""
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        if n_classes == 2:
            self.filters_ = self._fit_binary(X, y, self.classes_[0], self.classes_[1])
        else:
            all_filters = []
            n_comp_per_class = max(2, self.n_components // n_classes)
            for c in self.classes_:
                y_bin = (y == c).astype(int)
                filters = self._fit_binary(X, y_bin, 1, 0, n_comp_per_class)
                all_filters.append(filters)
            self.filters_ = np.vstack(all_filters)
        
        return self
    
    def transform(self, X: np.ndarray) -> np.ndarray:
        """Extract CSP features."""
        n_trials = X.shape[0]
        n_features = self.filters_.shape[0]
        features = np.zeros((n_trials, n_features))
        
        for i in range(n_trials):
            Z = self.filters_ @ X[i]
            var = np.var(Z, axis=1)
            var = var / np.sum(var)
            features[i] = np.log(var + 1e-10)
        
        return features
    
    def fit_transform(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        return self.fit(X, y).transform(X)
    
    def _fit_binary(self, X, y, c0, c1, n_comp=None):
        n_comp = n_comp or self.n_components
        X0 = X[y == c0]
        X1 = X[y == c1]
        
        C0 = self._class_cov(X0)
        C1 = self._class_cov(X1)
        
        C_comp = C0 + C1
        C_comp += self.reg * np.trace(C_comp) * np.eye(C_comp.shape[0])
        
        try:
            from scipy.linalg import eigh
            evals, evecs = eigh(C0, C_comp)
        except Exception:
            evals, evecs = np.linalg.eigh(np.linalg.solve(C_comp, C0))
        
        idx = np.argsort(evals)[::-1]
        evals = evals[idx]
        evecs = evecs[:, idx]
        
        n_first = n_comp // 2
        n_last = n_comp - n_first
        sel = np.concatenate([np.arange(n_first), np.arange(-n_last, 0)])
        
        return evecs[:, sel].T
    
    def _class_cov(self, X):
        covs = []
        for trial in X:
            c = np.cov(trial)
            c /= np.trace(c)
            covs.append(c)
        return np.mean(covs, axis=0)


class BandPowerExtractor:
    """Band power feature extractor."""
    
    def __init__(self, bands: dict = None, fs: float = 250):
        self.bands = bands or {'mu': (8, 12), 'beta_low': (12, 20), 'beta_high': (20, 30)}
        self.fs = fs
    
    def extract(self, X: np.ndarray) -> np.ndarray:
        """Extract band power features."""
        n_trials = X.shape[0]
        n_channels = X.shape[1]
        n_bands = len(self.bands)
        features = np.zeros((n_trials, n_channels * n_bands))
        
        for i in range(n_trials):
            feat_idx = 0
            for ch in range(n_channels):
                freqs, psd = scipy_signal.welch(
                    X[i, ch], fs=self.fs, 
                    nperseg=min(256, X.shape[2])
                )
                for _, (low, high) in self.bands.items():
                    mask = (freqs >= low) & (freqs <= high)
                    if np.any(mask):
                        bp = np.log10(np.mean(psd[mask]) + 1e-10)
                    else:
                        bp = -10
                    features[i, feat_idx] = bp
                    feat_idx += 1
        
        return features




# ============================================================================
# EEG DATA AUGMENTATION (for deep learning with small datasets)
# ============================================================================

class EEGDataAugmenter:
    """EEG-specific data augmentation for deep learning models.
    
    Techniques:
    - Time shift: random temporal jitter (simulates onset variability)
    - Gaussian noise: SNR-scaled noise injection
    - Channel dropout: randomly zero out channels (robustness)
    - Amplitude scaling: random gain variation
    """
    
    def __init__(self, time_shift_ms=50, noise_std_factor=0.1,
                 channel_drop_prob=0.1, amp_scale_range=(0.9, 1.1),
                 random_seed=42):
        self.time_shift_samples = int(time_shift_ms * 250 / 1000)  # at 250 Hz
        self.noise_std_factor = noise_std_factor
        self.channel_drop_prob = channel_drop_prob
        self.amp_scale_range = amp_scale_range
        self.rng = np.random.RandomState(random_seed)
    
    def augment(self, X, y, n_augmented=2):
        """Augment dataset by generating n_augmented copies per trial.
        
        Args:
            X: (n_trials, n_channels, n_samples) EEG data
            y: (n_trials,) labels
            n_augmented: number of augmented copies per original trial
            
        Returns:
            X_aug: (n_trials * (1 + n_augmented), n_channels, n_samples)
            y_aug: (n_trials * (1 + n_augmented),)
        """
        X_all = [X.copy()]
        y_all = [y.copy()]
        
        for _ in range(n_augmented):
            X_new = np.zeros_like(X)
            for i in range(len(X)):
                X_new[i] = self._augment_trial(X[i])
            X_all.append(X_new)
            y_all.append(y.copy())
        
        X_aug = np.concatenate(X_all, axis=0)
        y_aug = np.concatenate(y_all, axis=0)
        
        # Shuffle
        idx = self.rng.permutation(len(y_aug))
        return X_aug[idx], y_aug[idx]
    
    def _augment_trial(self, trial):
        """Apply random augmentations to a single trial (channels, samples)."""
        aug = trial.copy()
        
        # 1. Time shift (circular)
        shift = self.rng.randint(-self.time_shift_samples, self.time_shift_samples + 1)
        if shift != 0:
            aug = np.roll(aug, shift, axis=1)
        
        # 2. Gaussian noise injection
        noise_std = np.std(aug) * self.noise_std_factor
        aug += self.rng.randn(*aug.shape) * noise_std
        
        # 3. Channel dropout
        for ch in range(aug.shape[0]):
            if self.rng.rand() < self.channel_drop_prob:
                aug[ch] = 0.0
        
        # 4. Amplitude scaling
        scale = self.rng.uniform(*self.amp_scale_range)
        aug *= scale
        
        return aug


# ============================================================================
# CLASSIFIERS MODULE (Wrappers)
# ============================================================================

class ClassifierFactory:
    """Create and manage classifiers."""
    
    # Braindecode-based deep learning classifiers
    BRAINDECODE_CLASSIFIERS = ['atcnet', 'eegconformer', 'eegtcnet', 'ctnet']
    
    # All deep learning classifiers (for is_deep checks)
    DEEP_CLASSIFIERS = [
        'eegnet', 'shallow_convnet', 'deep_convnet',
        'atcnet', 'eegconformer', 'eegtcnet', 'ctnet',
        'mscformer',
    ]
    
    @staticmethod
    def create(name: str, config: dict = None, n_classes: int = 4, 
               n_channels: int = 22, n_samples: int = 1000,
               fold_id: int = 0) -> Any:
        config = config or {}
        
        if name == 'lda':
            return ClassifierFactory._create_lda(config, n_classes)
        elif name == 'svm':
            return ClassifierFactory._create_svm(config, n_classes)
        elif name == 'eegnet':
            return ClassifierFactory._create_eegnet(config, n_classes, n_channels, n_samples)
        elif name == 'shallow_convnet':
            return ClassifierFactory._create_shallow(config, n_classes, n_channels, n_samples)
        elif name == 'deep_convnet':
            return ClassifierFactory._create_deep(config, n_classes, n_channels, n_samples)
        elif name in ClassifierFactory.BRAINDECODE_CLASSIFIERS:
            return ClassifierFactory._create_braindecode(name, config, n_classes, n_channels, n_samples)
        elif name == 'mscformer':
            return ClassifierFactory._create_mscformer(config, n_classes, n_channels, n_samples)
        else:
            raise ValueError(f"Unknown classifier: {name}")
    
    @staticmethod
    def _create_lda(config, n_classes):
        from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
        return Pipeline([
            ('scaler', StandardScaler()),
            ('lda', LinearDiscriminantAnalysis(
                solver=config.get('solver', 'svd'),
                shrinkage=config.get('shrinkage', None),
            ))
        ])
    
    @staticmethod
    def _create_svm(config, n_classes):
        from sklearn.svm import SVC
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
        return Pipeline([
            ('scaler', StandardScaler()),
            ('svm', SVC(
                kernel=config.get('kernel', 'rbf'),
                C=config.get('C', 1.0),
                gamma=config.get('gamma', 'scale'),
                probability=True,
                random_state=42,
            ))
        ])
    
    @staticmethod
    def _create_eegnet(config, n_classes, n_channels, n_samples):
        try:
            import torch
            import torch.nn as nn
            
            class EEGNetModel(nn.Module):
                """Compact EEGNet implementation (Lawhern et al., 2018)."""
                def __init__(self, n_ch, n_samp, n_cls, F1=8, D=2, F2=16, dropout=0.5):
                    super().__init__()
                    # Block 1: Temporal + Spatial convolution
                    self.conv1 = nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False)
                    self.bn1 = nn.BatchNorm2d(F1)
                    self.depthwise = nn.Conv2d(F1, F1 * D, (n_ch, 1), groups=F1, bias=False)
                    self.bn2 = nn.BatchNorm2d(F1 * D)
                    self.pool1 = nn.AvgPool2d((1, 4))
                    self.drop1 = nn.Dropout(dropout)
                    
                    # Block 2: Separable convolution
                    self.separable1 = nn.Conv2d(F1 * D, F2, (1, 16), padding=(0, 8), bias=False)
                    self.bn3 = nn.BatchNorm2d(F2)
                    self.pool2 = nn.AvgPool2d((1, 8))
                    self.drop2 = nn.Dropout(dropout)
                    
                    # Compute flattened size
                    with torch.no_grad():
                        dummy = torch.zeros(1, 1, n_ch, n_samp)
                        out = self._forward_features(dummy)
                        flat_size = out.view(1, -1).size(1)
                    
                    self.fc = nn.Linear(flat_size, n_cls)
                
                def _forward_features(self, x):
                    x = self.conv1(x)
                    x = self.bn1(x)
                    x = self.depthwise(x)
                    x = self.bn2(x)
                    x = nn.functional.elu(x)
                    x = self.pool1(x)
                    x = self.drop1(x)
                    x = self.separable1(x)
                    x = self.bn3(x)
                    x = nn.functional.elu(x)
                    x = self.pool2(x)
                    x = self.drop2(x)
                    return x
                
                def forward(self, x):
                    if x.dim() == 3:
                        x = x.unsqueeze(1)
                    x = self._forward_features(x)
                    x = x.flatten(1)
                    return self.fc(x)
            
            model = EEGNetModel(
                n_ch=n_channels, n_samp=n_samples, n_cls=n_classes,
                F1=config.get('F1', 8), D=config.get('D', 2),
                F2=config.get('F2', 16), dropout=config.get('dropout_rate', 0.5),
            )
            return _TorchClassifierWrapper(
                model, n_classes=n_classes, n_channels=n_channels,
                n_samples=n_samples, config=config, name='eegnet',
                fold_id=fold_id,
            )
        except Exception as e:
            logger.warning(f"EEGNet creation failed ({e}), using feature-based SVM fallback")
            return ClassifierFactory._create_svm(config, n_classes)
    
    @staticmethod
    def _create_shallow(config, n_classes, n_channels, n_samples):
        """Shallow ConvNet - simplified wrapper."""
        try:
            import torch
            import torch.nn as nn
            
            class ShallowNet(nn.Module):
                def __init__(self, n_ch, n_samp, n_cls):
                    super().__init__()
                    self.conv_time = nn.Conv2d(1, 40, (1, 25), bias=False)
                    self.conv_spat = nn.Conv2d(40, 40, (n_ch, 1), bias=False)
                    self.bn = nn.BatchNorm2d(40)
                    self.pool = nn.AvgPool2d((1, 75), stride=(1, 15))
                    self.drop = nn.Dropout(0.5)
                    
                    with torch.no_grad():
                        dummy = torch.zeros(1, 1, n_ch, n_samp)
                        out = self.pool(self.bn(self.conv_spat(self.conv_time(dummy))))
                        flat_size = out.view(1, -1).size(1)
                    
                    self.fc = nn.Linear(flat_size, n_cls)
                
                def forward(self, x):
                    if x.dim() == 3:
                        x = x.unsqueeze(1)
                    x = self.conv_time(x)
                    x = self.conv_spat(x)
                    x = self.bn(x)
                    x = x ** 2  # Square activation
                    x = self.pool(x)
                    x = torch.log(torch.clamp(x, min=1e-6))
                    x = self.drop(x)
                    x = x.flatten(1)
                    return self.fc(x)
            
            return _TorchClassifierWrapper(
                ShallowNet(n_channels, n_samples, n_classes),
                n_classes=n_classes,
                n_channels=n_channels,
                n_samples=n_samples,
                config=config,
                name='shallow_convnet',
                fold_id=fold_id,
            )
        except ImportError:
            logger.warning("PyTorch not available for ShallowConvNet")
            return ClassifierFactory._create_svm(config, n_classes)
    
    @staticmethod
    def _create_deep(config, n_classes, n_channels, n_samples):
        """Deep ConvNet - simplified wrapper."""
        try:
            import torch
            import torch.nn as nn
            
            class DeepNet(nn.Module):
                def __init__(self, n_ch, n_samp, n_cls):
                    super().__init__()
                    self.block1 = nn.Sequential(
                        nn.Conv2d(1, 25, (1, 10), bias=False),
                        nn.Conv2d(25, 25, (n_ch, 1), bias=False),
                        nn.BatchNorm2d(25),
                        nn.ELU(),
                        nn.MaxPool2d((1, 3)),
                        nn.Dropout(0.5)
                    )
                    self.block2 = nn.Sequential(
                        nn.Conv2d(25, 50, (1, 10), bias=False, padding=(0, 5)),
                        nn.BatchNorm2d(50),
                        nn.ELU(),
                        nn.MaxPool2d((1, 3)),
                        nn.Dropout(0.5)
                    )
                    self.block3 = nn.Sequential(
                        nn.Conv2d(50, 100, (1, 10), bias=False, padding=(0, 5)),
                        nn.BatchNorm2d(100),
                        nn.ELU(),
                        nn.MaxPool2d((1, 3)),
                        nn.Dropout(0.5)
                    )
                    
                    with torch.no_grad():
                        dummy = torch.zeros(1, 1, n_ch, n_samp)
                        out = self.block3(self.block2(self.block1(dummy)))
                        flat_size = out.view(1, -1).size(1)
                    
                    self.fc = nn.Linear(flat_size, n_cls)
                
                def forward(self, x):
                    if x.dim() == 3:
                        x = x.unsqueeze(1)
                    x = self.block1(x)
                    x = self.block2(x)
                    x = self.block3(x)
                    x = x.flatten(1)
                    return self.fc(x)
            
            return _TorchClassifierWrapper(
                DeepNet(n_channels, n_samples, n_classes),
                n_classes=n_classes,
                n_channels=n_channels,
                n_samples=n_samples,
                config=config,
                name='deep_convnet',
                fold_id=fold_id,
            )
        except ImportError:
            logger.warning("PyTorch not available for DeepConvNet")
            return ClassifierFactory._create_svm(config, n_classes)


    @staticmethod
    def _create_braindecode(name, config, n_classes, n_channels, n_samples):
        """Create a braindecode-based classifier (ATCNet, EEGConformer, EEGTCNet, CTNet)."""
        try:
            import torch
            from braindecode.models import ATCNet, EEGConformer, EEGTCNet, CTNet
            
            model_map = {
                'atcnet': ATCNet,
                'eegconformer': EEGConformer,
                'eegtcnet': EEGTCNet,
                'ctnet': CTNet,
            }
            
            ModelClass = model_map[name]
            model = ModelClass(
                n_chans=n_channels,
                n_outputs=n_classes,
                n_times=n_samples,
            )
            
            return _TorchClassifierWrapper(
                model, n_classes=n_classes, n_channels=n_channels,
                n_samples=n_samples, config=config, name=name,
                fold_id=fold_id,
            )
        except Exception as e:
            logger.warning(f"{name} creation failed ({e}), using feature-based SVM fallback")
            return ClassifierFactory._create_svm(config, n_classes)
    
    @staticmethod
    def _create_mscformer(config, n_classes, n_channels, n_samples):
        """Create MSCFormer classifier (Zhao et al., 2025).
        
        Multi-Scale Convolutional Transformer for Motor Imagery BCI.
        Reference: https://github.com/snailpt/MSCFormer
        """
        try:
            import torch
            import torch.nn as nn
            import math
            from einops.layers.torch import Rearrange
            from einops import rearrange
            import torch.nn.functional as F
            
            class _MSCPatchEmbeddingCNN(nn.Module):
                """Multi-scale convolutional module for MSCFormer."""
                def __init__(self, f1=16, pooling_size=44, dropout_rate=0.5, number_channel=22):
                    super().__init__()
                    self.cnn1 = nn.Sequential(
                        nn.Conv2d(1, f1, (1, 85), (1, 1), padding='same'),
                        nn.Conv2d(f1, f1, (number_channel, 1), (1, 1), groups=f1),
                        nn.BatchNorm2d(f1), nn.ELU(),
                        nn.AvgPool2d((1, pooling_size)), nn.Dropout(dropout_rate),
                    )
                    self.cnn2 = nn.Sequential(
                        nn.Conv2d(1, f1, (1, 65), (1, 1), padding='same'),
                        nn.Conv2d(f1, f1, (number_channel, 1), (1, 1), groups=f1),
                        nn.BatchNorm2d(f1), nn.ELU(),
                        nn.AvgPool2d((1, pooling_size)), nn.Dropout(dropout_rate),
                    )
                    self.cnn3 = nn.Sequential(
                        nn.Conv2d(1, f1, (1, 45), (1, 1), padding='same'),
                        nn.Conv2d(f1, f1, (number_channel, 1), (1, 1), groups=f1),
                        nn.BatchNorm2d(f1), nn.ELU(),
                        nn.AvgPool2d((1, pooling_size)), nn.Dropout(dropout_rate),
                    )
                    self.projection = nn.Sequential(Rearrange('b e (h) (w) -> b (h w) e'))
                
                def forward(self, x):
                    x1, x2, x3 = self.cnn1(x), self.cnn2(x), self.cnn3(x)
                    x = torch.cat([x1, x2, x3], dim=1)
                    return self.projection(x)
            
            class _MSCMultiHeadAttention(nn.Module):
                def __init__(self, emb_size, num_heads, dropout):
                    super().__init__()
                    self.emb_size = emb_size
                    self.num_heads = num_heads
                    self.keys = nn.Linear(emb_size, emb_size)
                    self.queries = nn.Linear(emb_size, emb_size)
                    self.values = nn.Linear(emb_size, emb_size)
                    self.att_drop = nn.Dropout(dropout)
                    self.projection = nn.Linear(emb_size, emb_size)
                
                def forward(self, x, mask=None):
                    queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
                    keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
                    values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
                    energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
                    if mask is not None:
                        energy.mask_fill(~mask, torch.finfo(torch.float32).min)
                    scaling = self.emb_size ** (1 / 2)
                    att = F.softmax(energy / scaling, dim=-1)
                    att = self.att_drop(att)
                    out = torch.einsum('bhal, bhlv -> bhav', att, values)
                    out = rearrange(out, "b h n d -> b n (h d)")
                    return self.projection(out)
            
            class _MSCFeedForward(nn.Sequential):
                def __init__(self, emb_size, expansion=4, drop_p=0.5):
                    super().__init__(
                        nn.Linear(emb_size, expansion * emb_size),
                        nn.GELU(), nn.Dropout(drop_p),
                        nn.Linear(expansion * emb_size, emb_size),
                    )
            
            class _MSCResidualAdd(nn.Module):
                def __init__(self, fn, emb_size, drop_p):
                    super().__init__()
                    self.fn = fn
                    self.drop = nn.Dropout(drop_p)
                    self.layernorm = nn.LayerNorm(emb_size)
                def forward(self, x, **kwargs):
                    return self.layernorm(self.drop(self.fn(x, **kwargs)) + x)
            
            class _MSCTransformerBlock(nn.Sequential):
                def __init__(self, emb_size, num_heads=4, drop_p=0.5, expansion=4):
                    super().__init__(
                        _MSCResidualAdd(_MSCMultiHeadAttention(emb_size, num_heads, drop_p), emb_size, drop_p),
                        _MSCResidualAdd(_MSCFeedForward(emb_size, expansion, drop_p), emb_size, drop_p),
                    )
            
            class _MSCTransformerEncoder(nn.Sequential):
                def __init__(self, heads, depth, emb_size):
                    super().__init__(*[_MSCTransformerBlock(emb_size, heads) for _ in range(depth)])
            
            class _MSCPositionalEncoding(nn.Module):
                def __init__(self, embedding, length=100, dropout=0.1):
                    super().__init__()
                    self.dropout = nn.Dropout(dropout)
                    self.encoding = nn.Parameter(torch.randn(1, length, embedding))
                def forward(self, x):
                    x = x + self.encoding[:, :x.shape[1], :].to(x.device)
                    return self.dropout(x)
            
            class MSCFormerModel(nn.Module):
                """MSCFormer: Multi-Scale Convolutional Transformer (Zhao et al., 2025).
                
                Adapted from: https://github.com/snailpt/MSCFormer
                - Removed .cuda() calls for CPU/GPU compatibility
                - Uses device-agnostic tensor placement
                """
                def __init__(self, n_channels=22, n_classes=4, n_samples=1000,
                             f1=16, heads=8, depth=5, pooling_size=44, dropout_rate=0.5):
                    super().__init__()
                    self.emb_size = f1 * 3  # 48
                    self.cnn = _MSCPatchEmbeddingCNN(
                        f1=f1, pooling_size=pooling_size,
                        dropout_rate=dropout_rate, number_channel=n_channels,
                    )
                    self.position = _MSCPositionalEncoding(self.emb_size, dropout=0.1)
                    self.trans = _MSCTransformerEncoder(heads, depth, self.emb_size)
                    self.flatten = nn.Flatten()
                    self.classification = nn.Sequential(
                        nn.Dropout(0.25),
                        nn.Linear(self.emb_size, n_classes),
                    )
                
                def forward(self, x):
                    if x.dim() == 3:
                        x = x.unsqueeze(1)  # (B, C, T) -> (B, 1, C, T)
                    x = self.cnn(x)
                    b, l, e = x.shape
                    # Prepend learnable class token
                    cls_token = torch.zeros((b, 1, e), requires_grad=True, device=x.device)
                    x = torch.cat((cls_token, x), dim=1)
                    x = x * math.sqrt(self.emb_size)
                    x = self.position(x)
                    x = self.trans(x)
                    features = x[:, 0, :]  # class token
                    return self.classification(features)
            
            pooling_size = config.get('pooling_size', 44)  # 44 for BCI IV-2a, 52 for IV-2b
            dropout_rate = config.get('dropout_rate', 0.5)
            
            model = MSCFormerModel(
                n_channels=n_channels, n_classes=n_classes, n_samples=n_samples,
                f1=16, heads=8, depth=5,
                pooling_size=pooling_size, dropout_rate=dropout_rate,
            )
            
            return _TorchClassifierWrapper(
                model, n_classes=n_classes, n_channels=n_channels,
                n_samples=n_samples, config=config, name='mscformer',
                fold_id=fold_id,
            )
        except Exception as e:
            logger.warning(f"MSCFormer creation failed ({e}), using feature-based SVM fallback")
            return ClassifierFactory._create_svm(config, n_classes)


# ============================================================================
# EMA (Exponential Moving Average) for model weights -- Phase 3A
# ============================================================================

class EMAModel:
    """Exponential Moving Average of model parameters.
    
    Maintains a shadow copy of parameters updated as:
        shadow = decay * shadow + (1 - decay) * param
    
    Used for smoother, more generalizable inference weights.
    """
    
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow: Dict[str, torch.Tensor] = {}
        self.backup: Dict[str, torch.Tensor] = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self, model: nn.Module) -> None:
        """Update shadow params after optimizer.step()."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(
                    param.data, alpha=1.0 - self.decay
                )
    
    def apply_shadow(self, model: nn.Module) -> None:
        """Replace model params with shadow (for eval)."""
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])
    
    def restore(self, model: nn.Module) -> None:
        """Restore original params (after eval)."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}


class _TorchClassifierWrapper:
    """Sklearn-compatible wrapper for PyTorch classifiers.
    
    Supports:
    - fold_id for reproducible per-fold augmentation (Phase 1B)
    - Configurable gradient clipping (Phase 1C.10)
    - Gradient accumulation for large batch sizes (Phase 2B)
    - EMA model averaging (Phase 3A)
    - Internal data augmentation on training split only (Phase 1B)
    - Training history tracking for figures (Phase 1F)
    """
    
    def __init__(self, model: nn.Module, n_classes: int, n_channels: int,
                 n_samples: int, config: dict = None, name: str = 'torch',
                 fold_id: int = 0):
        self.model = model
        self.n_classes = n_classes
        self.n_channels = n_channels
        self.n_samples = n_samples
        self.config = config or {}
        self.name = name
        self.fold_id = fold_id
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self._is_fitted = False
        self.training_history: Dict[str, List[float]] = {
            'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
        }
    
    def fit(self, X: np.ndarray, y: np.ndarray, **kwargs) -> '_TorchClassifierWrapper':
        import math
        from torch.utils.data import DataLoader, TensorDataset
        from sklearn.model_selection import train_test_split
        
        epochs = self.config.get('epochs', 200)
        desired_batch = self.config.get('batch_size', 64)
        lr = self.config.get('learning_rate', 0.001)
        patience = self.config.get('patience', 30)
        weight_decay = self.config.get('weight_decay', 0)
        grad_clip = self.config.get('grad_clip', 1.0)
        use_ema = EXPERIMENT_CONFIG.get('training', {}).get('use_ema', True)
        warmup_epochs = 5
        
        # Phase 2B: Gradient accumulation for large batch sizes
        actual_batch = min(desired_batch, 32)
        accum_steps = max(1, desired_batch // actual_batch)
        
        # Ensure 3D
        if X.ndim == 2:
            X = X.reshape(-1, self.n_channels, self.n_samples)
        
        # Train/validation split for early stopping (85/15)
        try:
            X_tr, X_val, y_tr, y_val = train_test_split(
                X, y, test_size=0.15, stratify=y, random_state=42
            )
        except ValueError:
            X_tr, X_val, y_tr, y_val = train_test_split(
                X, y, test_size=0.15, random_state=42
            )
        
        # Phase 1B: Augment ONLY training data (not validation)
        augmenter = EEGDataAugmenter(random_seed=42 + self.fold_id * 1000)
        X_tr, y_tr = augmenter.augment(X_tr, y_tr, n_augmented=2)
        
        X_tr_t = torch.FloatTensor(X_tr.astype(np.float32)).to(self.device)
        y_tr_t = torch.LongTensor(y_tr).to(self.device)
        X_val_t = torch.FloatTensor(X_val.astype(np.float32)).to(self.device)
        y_val_t = torch.LongTensor(y_val).to(self.device)
        
        train_dataset = TensorDataset(X_tr_t, y_tr_t)
        train_loader = DataLoader(train_dataset, batch_size=actual_batch, shuffle=True, drop_last=False)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=lr, weight_decay=weight_decay
        )
        
        # Phase 3A: EMA
        ema: Optional[EMAModel] = None
        if use_ema:
            ema = EMAModel(self.model, decay=0.999)
        
        # Cosine annealing scheduler with linear warmup
        def lr_lambda(epoch: int) -> float:
            if epoch < warmup_epochs:
                return (epoch + 1) / warmup_epochs
            progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
            return 0.5 * (1 + math.cos(math.pi * progress))
        
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        
        best_val_loss = float('inf')
        best_model_state: Optional[Dict[str, torch.Tensor]] = None
        best_ema_shadow: Optional[Dict[str, torch.Tensor]] = None
        patience_counter = 0
        
        self.model.train()
        for epoch in range(epochs):
            # Training with gradient accumulation
            total_loss = 0.0
            total_correct = 0
            total_samples = 0
            optimizer.zero_grad()
            
            for step, (bx, by) in enumerate(train_loader):
                out = self.model(bx)
                loss = criterion(out, by) / accum_steps
                loss.backward()
                
                total_loss += loss.item() * accum_steps
                total_correct += (out.argmax(dim=1) == by).sum().item()
                total_samples += len(by)
                
                if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                    if grad_clip is not None:
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)
                    optimizer.step()
                    if ema is not None:
                        ema.update(self.model)
                    optimizer.zero_grad()
            
            scheduler.step()
            
            avg_train_loss = total_loss / max(1, len(train_loader))
            train_acc = total_correct / max(1, total_samples)
            
            # Validation
            self.model.eval()
            if ema is not None:
                ema.apply_shadow(self.model)
            with torch.no_grad():
                val_out = self.model(X_val_t)
                val_loss = criterion(val_out, y_val_t).item()
                val_acc = (val_out.argmax(dim=1) == y_val_t).float().mean().item()
            if ema is not None:
                ema.restore(self.model)
            self.model.train()
            
            self.training_history['train_loss'].append(avg_train_loss)
            self.training_history['val_loss'].append(val_loss)
            self.training_history['train_acc'].append(train_acc)
            self.training_history['val_acc'].append(val_acc)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = {k: v.clone() for k, v in self.model.state_dict().items()}
                if ema is not None:
                    best_ema_shadow = {k: v.clone() for k, v in ema.shadow.items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
        
        # Restore best model
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
        # Apply best EMA shadow for inference
        if ema is not None and best_ema_shadow is not None:
            for name, param in self.model.named_parameters():
                if param.requires_grad and name in best_ema_shadow:
                    param.data.copy_(best_ema_shadow[name])
        
        self._is_fitted = True
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        self.model.eval()
        if X.ndim == 2:
            X = X.reshape(-1, self.n_channels, self.n_samples)
        X_t = torch.FloatTensor(X.astype(np.float32)).to(self.device)
        with torch.no_grad():
            out = self.model(X_t)
            _, pred = torch.max(out, 1)
        return pred.cpu().numpy()
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        self.model.eval()
        if X.ndim == 2:
            X = X.reshape(-1, self.n_channels, self.n_samples)
        X_t = torch.FloatTensor(X.astype(np.float32)).to(self.device)
        with torch.no_grad():
            out = self.model(X_t)
            proba = torch.softmax(out, dim=1)
        return proba.cpu().numpy()
    
    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        pred = self.predict(X)
        return float(np.mean(pred == y))


# ============================================================================
# APA AGENT (Lightweight Integration)
# ============================================================================

class APAAgentLite:
    """Lightweight APA agent for the experiment pipeline."""
    
    def __init__(self, config: dict):
        self.config = config.get('apa', {})
        bins = self.config.get('state_bins', EXPERIMENT_CONFIG['apa']['state_bins'])
        
        n_bins = [len(v) - 1 for v in bins.values()]
        self.n_states = int(np.prod(n_bins))
        self.n_actions = 3
        self.action_names = ['conservative', 'moderate', 'aggressive']
        
        lr = self.config.get('learning_rate', 0.1)
        self.q_table = np.zeros((self.n_states, self.n_actions))
        self.lr = lr
        self.gamma = self.config.get('discount_factor', 0.99)
        self.epsilon = self.config.get('epsilon_start', 1.0)
        self.epsilon_decay = self.config.get('epsilon_decay', 0.995)
        self.epsilon_min = self.config.get('epsilon_min', 0.01)
        
        self.bin_edges = bins
        self.feature_names = list(bins.keys())
        self.n_bins = n_bins
        
        self.action_counts = {0: 0, 1: 0, 2: 0}
        self.episode_rewards = []
        self.rng = np.random.RandomState(42)
    
    def discretize(self, state: dict) -> int:
        """Convert continuous state to discrete index."""
        indices = []
        for i, feat in enumerate(self.feature_names):
            val = state.get(feat, 0)
            edges = self.bin_edges[feat]
            bin_idx = np.digitize(val, edges[1:])
            bin_idx = min(bin_idx, self.n_bins[i] - 1)
            indices.append(bin_idx)
        
        # Row-major index
        idx = 0
        for i, b in enumerate(indices):
            multiplier = int(np.prod(self.n_bins[i+1:]))
            idx += b * multiplier
        return min(idx, self.n_states - 1)
    
    def select_action(self, state: dict) -> int:
        """Epsilon-greedy action selection."""
        state_idx = self.discretize(state)
        
        if self.rng.rand() < self.epsilon:
            action = self.rng.randint(self.n_actions)
        else:
            action = int(np.argmax(self.q_table[state_idx]))
        
        self.action_counts[action] += 1
        return action
    
    def update(self, state: dict, action: int, reward: float, 
               next_state: dict, done: bool = True):
        """Q-learning update."""
        s = self.discretize(state)
        s_next = self.discretize(next_state)
        
        best_next = np.max(self.q_table[s_next])
        td_target = reward + (0 if done else self.gamma * best_next)
        td_error = td_target - self.q_table[s, action]
        self.q_table[s, action] += self.lr * td_error
        
        if done:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
            self.episode_rewards.append(reward)
        
        return td_error
    
    def get_action_name(self, action: int) -> str:
        return self.action_names[action]
    
    def get_statistics(self) -> dict:
        return {
            'epsilon': self.epsilon,
            'action_distribution': {
                self.action_names[a]: c for a, c in self.action_counts.items()
            },
            'n_episodes': len(self.episode_rewards),
            'avg_reward': np.mean(self.episode_rewards) if self.episode_rewards else 0,
            'q_table_coverage': np.sum(np.any(self.q_table != 0, axis=1)) / self.n_states,
        }


# ============================================================================
# DVA AGENT (Lightweight Integration)
# ============================================================================

class DVAAgentLite:
    """Lightweight DVA agent for the experiment pipeline."""
    
    def __init__(self, config: dict):
        dva_config = config.get('dva', {})
        self.accept_threshold = dva_config.get('accept_threshold', 0.8)
        self.reject_threshold = dva_config.get('reject_threshold', 0.5)
        
        self.decision_counts = {'accept': 0, 'reject': 0, 'review': 0}
        self.correct_decisions = 0
        self.total_verified = 0
    
    def validate(self, probabilities: np.ndarray, true_label: int = None,
                signal_quality: float = 0.7) -> dict:
        """
        Validate a classification decision.
        
        Args:
            probabilities: Class probability vector
            true_label: True class label (for evaluation)
            signal_quality: Signal quality score [0,1]
        """
        confidence = float(np.max(probabilities))
        prediction = int(np.argmax(probabilities))
        
        # Margin between top 2
        sorted_probs = np.sort(probabilities)[::-1]
        margin = sorted_probs[0] - sorted_probs[1] if len(sorted_probs) > 1 else 0
        
        # Multi-validator scoring
        confidence_score = confidence
        margin_score = min(1.0, margin / 0.3)
        quality_score = signal_quality
        
        # Weighted aggregate
        agg_score = 0.4 * confidence_score + 0.3 * margin_score + 0.3 * quality_score
        
        # Decision
        if agg_score >= self.accept_threshold:
            decision = 'accept'
        elif agg_score < self.reject_threshold:
            decision = 'reject'
        else:
            decision = 'review'
        
        self.decision_counts[decision] += 1
        
        # Verify if true label provided
        correct = None
        if true_label is not None:
            if decision == 'accept':
                correct = (prediction == true_label)
            elif decision == 'reject':
                correct = (prediction != true_label)
            else:
                correct = True  # Review is always safe
            
            if correct:
                self.correct_decisions += 1
            self.total_verified += 1
        
        return {
            'decision': decision,
            'prediction': prediction,
            'confidence': confidence,
            'margin': margin,
            'agg_score': agg_score,
            'correct': correct,
            'signal_quality': signal_quality,
        }
    
    def get_statistics(self) -> dict:
        total = sum(self.decision_counts.values())
        return {
            'decision_counts': self.decision_counts.copy(),
            'accept_rate': self.decision_counts['accept'] / max(1, total),
            'reject_rate': self.decision_counts['reject'] / max(1, total),
            'review_rate': self.decision_counts['review'] / max(1, total),
            'verification_accuracy': self.correct_decisions / max(1, self.total_verified),
            'total_decisions': total,
        }
    
    def reset(self):
        self.decision_counts = {'accept': 0, 'reject': 0, 'review': 0}
        self.correct_decisions = 0
        self.total_verified = 0


# ============================================================================
# EVALUATION MODULE
# ============================================================================

class EvaluationMetrics:
    """Comprehensive evaluation metrics for BCI."""
    
    @staticmethod
    def compute_all(y_true: np.ndarray, y_pred: np.ndarray, 
                   y_proba: np.ndarray = None,
                   class_names: list = None) -> dict:
        """Compute all evaluation metrics."""
        from sklearn.metrics import (
            accuracy_score, cohen_kappa_score,
            precision_score, recall_score, f1_score,
            confusion_matrix, classification_report
        )
        
        n_classes = len(np.unique(y_true))
        
        acc = accuracy_score(y_true, y_pred)
        kappa = cohen_kappa_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        cm = confusion_matrix(y_true, y_pred)
        
        # Per-class metrics
        per_class = {}
        for c in range(n_classes):
            mask = y_true == c
            if np.sum(mask) > 0:
                c_name = class_names[c] if class_names and c < len(class_names) else f'Class {c}'
                per_class[c_name] = {
                    'accuracy': float(np.mean(y_pred[mask] == c)),
                    'n_samples': int(np.sum(mask)),
                    'precision': float(precision_score(y_true == c, y_pred == c, zero_division=0)),
                    'recall': float(recall_score(y_true == c, y_pred == c, zero_division=0)),
                    'f1': float(f1_score(y_true == c, y_pred == c, zero_division=0)),
                }
        
        result = {
            'accuracy': float(acc),
            'kappa': float(kappa),
            'precision_macro': float(precision),
            'recall_macro': float(recall),
            'f1_macro': float(f1),
            'confusion_matrix': cm.tolist(),
            'per_class': per_class,
            'n_samples': len(y_true),
        }
        
        return result
    
    @staticmethod
    def wilcoxon_test(scores_a: list, scores_b: list, 
                     alternative: str = 'two-sided') -> dict:
        """Wilcoxon signed-rank test with safety checks."""
        try:
            a = np.array(scores_a, dtype=float)
            b = np.array(scores_b, dtype=float)
            diff = a - b
            
            # Safety check: if all differences are zero, test is undefined
            if np.all(diff == 0):
                return {
                    'statistic': 0.0,
                    'p_value': 1.0,
                    'significant_005': False,
                    'significant_001': False,
                    'note': 'All differences are zero; test undefined',
                }
            
            # Remove zero differences for Wilcoxon (pratt method handles this)
            stat, p_value = scipy_stats.wilcoxon(
                a, b, alternative=alternative, zero_method='zsplit'
            )
            return {
                'statistic': float(stat),
                'p_value': float(p_value),
                'significant_005': p_value < 0.05,
                'significant_001': p_value < 0.01,
            }
        except Exception as e:
            return {'error': str(e), 'p_value': 1.0, 'significant_005': False, 'significant_001': False}
    
    @staticmethod
    def mcnemar_test(y_true, y_pred_a, y_pred_b) -> dict:
        """McNemar's test for comparing two classifiers."""
        correct_a = (y_pred_a == y_true)
        correct_b = (y_pred_b == y_true)
        
        # Contingency table
        b = np.sum(correct_a & ~correct_b)  # A correct, B wrong
        c = np.sum(~correct_a & correct_b)  # A wrong, B correct
        
        if b + c == 0:
            return {'statistic': 0, 'p_value': 1.0, 'significant_005': False}
        
        # McNemar with continuity correction
        stat = (abs(b - c) - 1) ** 2 / (b + c)
        p_value = 1 - scipy_stats.chi2.cdf(stat, df=1)
        
        return {
            'statistic': float(stat),
            'p_value': float(p_value),
            'b': int(b),
            'c': int(c),
            'significant_005': p_value < 0.05,
        }


# ============================================================================
# EXPERIMENT RUNNER
# ============================================================================

class ExperimentRunner:
    """
    End-to-end experiment runner for LLM-EEG Framework.
    
    Orchestrates:
    1. Data generation/loading
    2. Preprocessing (with/without APA)
    3. Feature extraction
    4. Classification
    5. Decision validation (DVA)
    6. Evaluation and reporting
    """
    
    def __init__(self, config: dict = None, output_dir: str = None):
        self.config = config or EXPERIMENT_CONFIG
        self.output_dir = Path(output_dir or 'results')
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.preprocessor = EEGPreprocessor(self.config)
        self.csp = CSPFeatureExtractor(
            n_components=self.config['features']['csp_components'],
            reg=self.config['features']['csp_reg'],
        )
        self.band_power = BandPowerExtractor(
            bands=self.config['features']['band_power_bands'],
            fs=self.config['dataset']['sampling_rate'],
        )
        self.apa = APAAgentLite(self.config)
        self.dva = DVAAgentLite(self.config)
        
        # Compute MI samples from config
        self.mi_samples_config = int(
            self.config['dataset']['sampling_rate'] * self.config['dataset']['trial_duration']
        )
        
        # Results storage
        self.results = {
            'subjects': {},
            'summary': {},
            'apa_stats': {},
            'dva_stats': {},
            'statistical_tests': {},
            'timestamp': datetime.now().isoformat(),
            'config': self.config,
        }
    
    def run_synthetic_experiment(self, n_subjects: int = 9, 
                                 n_trials_per_class: int = 72,
                                 classifiers: list = None):
        """Run complete experiment with synthetic data."""
        logger.info("=" * 70)
        logger.info("LLM-EEG Framework: End-to-End Experiment")
        logger.info("=" * 70)
        logger.info(f"Mode: Synthetic Data")
        logger.info(f"Subjects: {n_subjects}")
        logger.info(f"Trials/class: {n_trials_per_class}")
        
        if classifiers is None:
            classifiers = ['lda', 'svm']
            try:
                import torch
                classifiers.extend(['eegnet', 'shallow_convnet', 'deep_convnet'])
            except ImportError:
                logger.warning("PyTorch not available. Using only traditional classifiers.")
        
        logger.info(f"Classifiers: {classifiers}")
        
        generator = SyntheticBCI2aGenerator(self.config)
        
        all_subject_results = {}
        
        for subj in range(1, n_subjects + 1):
            logger.info(f"\n{'='*50}")
            logger.info(f"Subject {subj}/{n_subjects}")
            logger.info(f"{'='*50}")
            
            # B1: Reset APA for each subject (prevent cross-subject interference)
            self.apa = APAAgentLite(self.config)
            subj_results = self._run_subject(
                subj, generator, n_trials_per_class, classifiers
            )
            all_subject_results[f'S{subj:02d}'] = subj_results
        
        self.results['subjects'] = all_subject_results
        
        # Compute summary
        self._compute_summary(classifiers)
        
        # Statistical tests
        self._compute_statistical_tests(classifiers)
        
        # Save results
        self._save_results()
        
        # Print summary
        self._print_summary()
        
        return self.results
    
    def run_real_experiment(self, data_dir: str,
                            classifiers: list = None,
                            n_folds: int = 5):
        """Run experiment with real BCI IV-2a .mat data."""
        logger.info("=" * 70)
        logger.info("LLM-EEG Framework: End-to-End Experiment (REAL DATA)")
        logger.info("=" * 70)
        logger.info(f"Data directory: {data_dir}")
        
        loader = RealBCI2aLoader(data_dir, self.config)
        
        if not loader.is_available():
            raise FileNotFoundError(f"No BCI IV-2a .mat files found in {data_dir}")
        
        subjects = loader.get_available_subjects()
        logger.info(f"Available subjects: {subjects}")
        
        if classifiers is None:
            classifiers = ['lda', 'svm']
            try:
                import torch
                classifiers.extend(['eegnet', 'shallow_convnet', 'deep_convnet'])
            except ImportError:
                logger.warning("PyTorch not available. Using only traditional classifiers.")
        
        logger.info(f"Classifiers: {classifiers}")
        logger.info(f"Cross-validation folds: {n_folds}")
        
        all_subject_results = {}
        
        for subj in subjects:
            logger.info(f"\n{'='*50}")
            logger.info(f"Subject {subj} (Real Data)")
            logger.info(f"{'='*50}")
            
            try:
                X_train, y_train, X_test, y_test = loader.load_both_sessions(subj, mi_period_only=True)
                
                if len(X_train) == 0:
                    logger.warning(f"  No valid trials for subject {subj}, skipping")
                    continue
                
                # Reset APA for each subject (prevent cross-subject interference)
                self.apa = APAAgentLite(self.config)
                subj_results = self._run_subject_real(
                    subj, X_train, y_train, classifiers, n_folds,
                    X_test=X_test, y_test=y_test,
                )
                all_subject_results[f'S{subj:02d}'] = subj_results
                
            except Exception as e:
                logger.error(f"  Error processing subject {subj}: {e}")
                all_subject_results[f'S{subj:02d}'] = {'error': str(e)}
        
        self.results['subjects'] = all_subject_results
        self.results['data_source'] = 'real_bci_iv_2a'
        self.results['data_dir'] = str(data_dir)
        self.results['dataset_info'] = loader.get_dataset_info()
        
        # Compute summary
        self._compute_summary(classifiers)
        self._compute_statistical_tests(classifiers)
        self._save_results()
        self._print_summary()
        
        return self.results
    
    def _run_subject_session_wise(self, subject_id: int,
                                 X_train: np.ndarray, y_train: np.ndarray,
                                 X_test: np.ndarray, y_test: np.ndarray,
                                 classifiers: List[str]) -> dict:
        """Run session-wise evaluation: train on T session, test on E session.
        
        Phase 2A: This is the standard protocol used by most published methods.
        """
        logger.info(f"  Session-wise: Train={X_train.shape}, Test={X_test.shape}")
        logger.info(f"  Train classes: {dict(zip(*np.unique(y_train, return_counts=True)))}")
        logger.info(f"  Test classes: {dict(zip(*np.unique(y_test, return_counts=True)))}")
        
        subject_results: Dict[str, dict] = {
            'n_train': len(y_train),
            'n_test': len(y_test),
            'eval_protocol': 'session-wise',
            'classifiers': {},
        }
        
        apa_deep_disabled = not self.config.get('apa', {}).get('deep_model_enabled', False)
        
        for clf_name in classifiers:
            is_deep_clf = clf_name in ClassifierFactory.DEEP_CLASSIFIERS
            
            # Phase 1D: Skip APA for deep models when disabled
            if is_deep_clf and apa_deep_disabled:
                conditions = ['baseline']
            else:
                conditions = ['baseline', 'apa']
            
            for condition in conditions:
                key = f'{clf_name}_{condition}'
                
                # Phase 1A: Route deep models to 'deep' profile
                if condition == 'baseline':
                    profile = 'deep' if is_deep_clf else 'moderate'
                    X_tr_pp = self.preprocessor.process(X_train, profile)
                    X_te_pp = self.preprocessor.process(X_test, profile)
                else:
                    # APA condition (only for traditional classifiers when reaching here)
                    if is_deep_clf:
                        # Session-level APA for deep models
                        sample_indices = np.random.choice(len(X_train), min(20, len(X_train)), replace=False)
                        avg_sq: Dict[str, float] = {}
                        for feat in ['snr', 'artifact_ratio', 'line_noise', 'signal_quality_score']:
                            vals = [self.preprocessor.compute_signal_quality(X_train[si:si+1])[feat]
                                    for si in sample_indices]
                            avg_sq[feat] = float(np.mean(vals))
                        action = self.apa.select_action(avg_sq)
                        # Force deep profile regardless of APA selection
                        X_tr_pp = self.preprocessor.process(X_train, 'deep')
                        X_te_pp = self.preprocessor.process(X_test, 'deep')
                        sq_after: Dict[str, float] = {}
                        for feat in ['snr', 'artifact_ratio', 'line_noise', 'signal_quality_score']:
                            vals = [self.preprocessor.compute_signal_quality(X_tr_pp[si:si+1])[feat]
                                    for si in sample_indices]
                            sq_after[feat] = float(np.mean(vals))
                        reward = (sq_after['signal_quality_score'] - avg_sq['signal_quality_score']) * 2
                        self.apa.update(avg_sq, action, reward, sq_after, done=True)
                    else:
                        # Trial-level APA for traditional classifiers
                        X_tr_pp = np.zeros_like(X_train)
                        for i in range(len(X_train)):
                            sq = self.preprocessor.compute_signal_quality(X_train[i:i+1])
                            action = self.apa.select_action(sq)
                            profile = self.apa.get_action_name(action)
                            X_tr_pp[i] = self.preprocessor.process(X_train[i:i+1], profile)[0]
                            sq_after_t = self.preprocessor.compute_signal_quality(X_tr_pp[i:i+1])
                            reward = (sq_after_t['signal_quality_score'] - sq['signal_quality_score']) * 2
                            self.apa.update(sq, action, reward, sq_after_t, done=True)
                        X_te_pp = np.zeros_like(X_test)
                        for i in range(len(X_test)):
                            sq = self.preprocessor.compute_signal_quality(X_test[i:i+1])
                            action = self.apa.select_action(sq)
                            profile = self.apa.get_action_name(action)
                            X_te_pp[i] = self.preprocessor.process(X_test[i:i+1], profile)[0]
                
                # Feature extraction
                if is_deep_clf:
                    X_tr_feat = np.zeros((len(y_train), 1))
                    X_te_feat = np.zeros((len(y_test), 1))
                else:
                    csp_sw = CSPFeatureExtractor(
                        n_components=self.config['features']['csp_components'],
                        reg=self.config['features']['csp_reg']
                    )
                    X_tr_csp = csp_sw.fit_transform(X_tr_pp, y_train)
                    X_te_csp = csp_sw.transform(X_te_pp)
                    bp_tr = self.band_power.extract(X_tr_pp)
                    bp_te = self.band_power.extract(X_te_pp)
                    X_tr_feat = np.hstack([X_tr_csp, bp_tr])
                    X_te_feat = np.hstack([X_te_csp, bp_te])
                
                try:
                    metrics = self._train_and_evaluate(
                        clf_name, X_tr_feat, y_train,
                        X_te_feat, y_test,
                        X_tr_pp, X_te_pp,
                        condition=condition,
                    )
                    subject_results['classifiers'][key] = metrics
                    logger.info(f"  {key}: Acc={metrics['accuracy']:.4f}, Kappa={metrics['kappa']:.4f}")
                except Exception as e:
                    logger.error(f"  {key} error: {e}")
                    subject_results['classifiers'][key] = {'accuracy': 0, 'kappa': 0, 'error': str(e)}
            
            # Phase 1D: Copy baseline to apa key if APA was skipped
            if is_deep_clf and apa_deep_disabled:
                baseline_key = f'{clf_name}_baseline'
                apa_key = f'{clf_name}_apa'
                if baseline_key in subject_results['classifiers'] and apa_key not in subject_results['classifiers']:
                    subject_results['classifiers'][apa_key] = subject_results['classifiers'][baseline_key].copy()
                    subject_results['classifiers'][apa_key]['condition'] = 'apa (copied from baseline)'
        
        # DVA analysis on test session
        self.dva.reset()
        best_clf_name = None
        best_acc = 0.0
        for clf_name in classifiers:
            for cond in ['baseline', 'apa']:
                key = f'{clf_name}_{cond}'
                if key in subject_results['classifiers']:
                    acc = subject_results['classifiers'][key].get('accuracy', 0)
                    if acc > best_acc:
                        best_acc = acc
                        best_clf_name = clf_name
        
        if best_clf_name:
            is_dl = best_clf_name in ClassifierFactory.DEEP_CLASSIFIERS
            profile = 'deep' if is_dl else 'moderate'
            X_tr_pp_dva = self.preprocessor.process(X_train, profile)
            X_te_pp_dva = self.preprocessor.process(X_test, profile)
            
            try:
                clf = ClassifierFactory.create(
                    best_clf_name,
                    self.config['classifiers'].get(best_clf_name, {}),
                    n_classes=self.config['dataset']['n_classes'],
                    n_channels=self.config['dataset']['n_channels'],
                    n_samples=self.mi_samples_config,
                )
                actually_deep = is_dl and isinstance(clf, _TorchClassifierWrapper)
                
                if actually_deep:
                    clf.fit(X_tr_pp_dva, y_train)
                    probas = clf.predict_proba(X_te_pp_dva)
                else:
                    csp_dva = CSPFeatureExtractor(
                        n_components=self.config['features']['csp_components'],
                        reg=self.config['features']['csp_reg']
                    )
                    X_tr_feat_dva = csp_dva.fit_transform(X_tr_pp_dva, y_train)
                    X_te_feat_dva = csp_dva.transform(X_te_pp_dva)
                    bp_tr_dva = self.band_power.extract(X_tr_pp_dva)
                    bp_te_dva = self.band_power.extract(X_te_pp_dva)
                    clf.fit(np.hstack([X_tr_feat_dva, bp_tr_dva]), y_train)
                    probas = clf.predict_proba(np.hstack([X_te_feat_dva, bp_te_dva]))
                
                dva_results_list: List[dict] = []
                for i in range(len(y_test)):
                    sq = self.preprocessor.compute_signal_quality(X_te_pp_dva[i:i+1])
                    dva_result = self.dva.validate(
                        probas[i], true_label=y_test[i],
                        signal_quality=sq['signal_quality_score']
                    )
                    dva_results_list.append(dva_result)
                
                if dva_results_list:
                    subject_results['dva_results'] = {
                        'statistics': self.dva.get_statistics(),
                        'n_accepted': sum(1 for r in dva_results_list if r['decision'] == 'accept'),
                        'n_rejected': sum(1 for r in dva_results_list if r['decision'] == 'reject'),
                        'n_reviewed': sum(1 for r in dva_results_list if r['decision'] == 'review'),
                        'accepted_accuracy': float(np.mean([
                            r['prediction'] == y_test[i]
                            for i, r in enumerate(dva_results_list)
                            if r['decision'] == 'accept'
                        ])) if any(r['decision'] == 'accept' for r in dva_results_list) else 0,
                    }
            except Exception as e:
                logger.warning(f"  DVA error: {e}")
        
        subject_results['apa_stats'] = self.apa.get_statistics()
        return subject_results

    def _run_subject_real(self, subject_id: int, X: np.ndarray, y: np.ndarray,
                          classifiers: List[str], n_folds: int = 5,
                          X_test: Optional[np.ndarray] = None,
                          y_test: Optional[np.ndarray] = None) -> dict:
        """Run experiment for a single subject using real data.
        
        If X_test/y_test are provided (evaluation session), uses session-wise protocol.
        Otherwise, falls back to stratified k-fold CV on training session only.
        """
        # Phase 2A: Session-wise evaluation if E session available
        if X_test is not None and y_test is not None:
            return self._run_subject_session_wise(subject_id, X, y, X_test, y_test, classifiers)
        
        from sklearn.model_selection import StratifiedKFold
        
        logger.info(f"  Fallback: k-fold CV (no evaluation session)")
        logger.info(f"  Data shape: X={X.shape}, y={y.shape}")
        logger.info(f"  Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
        
        subject_results: Dict[str, dict] = {
            'n_trials': len(y),
            'eval_protocol': f'{n_folds}-fold CV',
            'classifiers': {},
        }
        
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42 + subject_id)
        apa_deep_disabled = not self.config.get('apa', {}).get('deep_model_enabled', False)
        
        for clf_name in classifiers:
            is_deep_clf = clf_name in ClassifierFactory.DEEP_CLASSIFIERS
            
            # Phase 1D: Skip APA for deep models when disabled
            if is_deep_clf and apa_deep_disabled:
                conditions = ['baseline']
            else:
                conditions = ['baseline', 'apa']
            
            for condition in conditions:
                key = f'{clf_name}_{condition}'
                fold_metrics: List[dict] = []
                
                for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                    X_train, y_train = X[train_idx], y[train_idx]
                    X_test_fold, y_test_fold = X[test_idx], y[test_idx]
                    
                    # Phase 1A: Route deep models to 'deep' profile
                    if condition == 'baseline':
                        profile = 'deep' if is_deep_clf else 'moderate'
                        X_train_pp = self.preprocessor.process(X_train, profile)
                        X_test_pp = self.preprocessor.process(X_test_fold, profile)
                    elif condition == 'apa':
                        if is_deep_clf:
                            # Session-level APA with deep profile forced
                            sample_indices = np.random.choice(len(X_train), min(20, len(X_train)), replace=False)
                            avg_sq: Dict[str, float] = {}
                            for feat in ['snr', 'artifact_ratio', 'line_noise', 'signal_quality_score']:
                                vals = [self.preprocessor.compute_signal_quality(X_train[si:si+1])[feat]
                                        for si in sample_indices]
                                avg_sq[feat] = float(np.mean(vals))
                            action = self.apa.select_action(avg_sq)
                            X_train_pp = self.preprocessor.process(X_train, 'deep')
                            X_test_pp = self.preprocessor.process(X_test_fold, 'deep')
                            sq_after: Dict[str, float] = {}
                            for feat in ['snr', 'artifact_ratio', 'line_noise', 'signal_quality_score']:
                                vals = [self.preprocessor.compute_signal_quality(X_train_pp[si:si+1])[feat]
                                        for si in sample_indices]
                                sq_after[feat] = float(np.mean(vals))
                            reward = (sq_after['signal_quality_score'] - avg_sq['signal_quality_score']) * 2
                            self.apa.update(avg_sq, action, reward, sq_after, done=True)
                        else:
                            # Trial-level APA for traditional classifiers
                            X_train_pp = np.zeros_like(X_train)
                            for i in range(len(X_train)):
                                sq = self.preprocessor.compute_signal_quality(X_train[i:i+1])
                                action = self.apa.select_action(sq)
                                profile = self.apa.get_action_name(action)
                                X_train_pp[i] = self.preprocessor.process(X_train[i:i+1], profile)[0]
                                sq_after_t = self.preprocessor.compute_signal_quality(X_train_pp[i:i+1])
                                reward = (sq_after_t['signal_quality_score'] - sq['signal_quality_score']) * 2
                                self.apa.update(sq, action, reward, sq_after_t, done=True)
                            X_test_pp = np.zeros_like(X_test_fold)
                            for i in range(len(X_test_fold)):
                                sq = self.preprocessor.compute_signal_quality(X_test_fold[i:i+1])
                                action = self.apa.select_action(sq)
                                profile = self.apa.get_action_name(action)
                                X_test_pp[i] = self.preprocessor.process(X_test_fold[i:i+1], profile)[0]
                    
                    # Feature extraction
                    if is_deep_clf:
                        X_train_combined = np.zeros((len(y_train), 1))
                        X_test_combined = np.zeros((len(y_test_fold), 1))
                    else:
                        csp_fold = CSPFeatureExtractor(
                            n_components=self.config['features']['csp_components'],
                            reg=self.config['features']['csp_reg']
                        )
                        X_train_feat = csp_fold.fit_transform(X_train_pp, y_train)
                        X_test_feat = csp_fold.transform(X_test_pp)
                        bp_train = self.band_power.extract(X_train_pp)
                        bp_test = self.band_power.extract(X_test_pp)
                        X_train_combined = np.hstack([X_train_feat, bp_train])
                        X_test_combined = np.hstack([X_test_feat, bp_test])
                    
                    try:
                        metrics = self._train_and_evaluate(
                            clf_name, X_train_combined, y_train,
                            X_test_combined, y_test_fold,
                            X_train_pp, X_test_pp,
                            condition=condition,
                            fold_id=fold_idx,
                        )
                        fold_metrics.append(metrics)
                    except Exception as e:
                        logger.error(f"    Fold {fold_idx+1} error ({key}): {e}")
                        continue
                
                if fold_metrics:
                    avg_result = {
                        'accuracy': float(np.mean([m['accuracy'] for m in fold_metrics])),
                        'kappa': float(np.mean([m['kappa'] for m in fold_metrics])),
                        'precision_macro': float(np.mean([m.get('precision_macro', 0) for m in fold_metrics])),
                        'recall_macro': float(np.mean([m.get('recall_macro', 0) for m in fold_metrics])),
                        'f1_macro': float(np.mean([m.get('f1_macro', 0) for m in fold_metrics])),
                        'accuracy_std': float(np.std([m['accuracy'] for m in fold_metrics])),
                        'kappa_std': float(np.std([m['kappa'] for m in fold_metrics])),
                        'n_folds': len(fold_metrics),
                        'confusion_matrix': fold_metrics[-1].get('confusion_matrix', []),
                        'per_class': fold_metrics[-1].get('per_class', {}),
                        'classifier': clf_name,
                        'condition': condition,
                    }
                    subject_results['classifiers'][key] = avg_result
                    logger.info(f"  {key}: Acc={avg_result['accuracy']:.4f}+/-{avg_result['accuracy_std']:.4f}, "
                              f"Kappa={avg_result['kappa']:.4f}")
                else:
                    subject_results['classifiers'][key] = {
                        'accuracy': 0, 'kappa': 0, 'error': 'All folds failed'
                    }
            
            # Phase 1D: Copy baseline to apa key if APA was skipped
            if is_deep_clf and apa_deep_disabled:
                baseline_key = f'{clf_name}_baseline'
                apa_key = f'{clf_name}_apa'
                if baseline_key in subject_results['classifiers'] and apa_key not in subject_results['classifiers']:
                    subject_results['classifiers'][apa_key] = subject_results['classifiers'][baseline_key].copy()
                    subject_results['classifiers'][apa_key]['condition'] = 'apa (copied from baseline)'
        
        # DVA analysis across all folds
        self.dva.reset()
        best_clf_name = None
        best_acc = 0.0
        for clf_name in classifiers:
            key = f'{clf_name}_apa'
            if key in subject_results['classifiers']:
                acc = subject_results['classifiers'][key].get('accuracy', 0)
                if acc > best_acc:
                    best_acc = acc
                    best_clf_name = clf_name
        
        if best_clf_name:
            all_dva_results: List[dict] = []
            all_y_test_dva: List[int] = []
            
            is_dl = best_clf_name in ClassifierFactory.DEEP_CLASSIFIERS
            profile = 'deep' if is_dl else 'moderate'
            
            for fold_idx, (train_idx_f, test_idx_f) in enumerate(skf.split(X, y)):
                X_train_f = X[train_idx_f]
                y_train_f = y[train_idx_f]
                X_test_f = X[test_idx_f]
                y_test_f = y[test_idx_f]
                X_train_pp_f = self.preprocessor.process(X_train_f, profile)
                X_test_pp_f = self.preprocessor.process(X_test_f, profile)
                
                try:
                    clf = ClassifierFactory.create(
                        best_clf_name,
                        self.config['classifiers'].get(best_clf_name, {}),
                        n_classes=self.config['dataset']['n_classes'],
                        n_channels=self.config['dataset']['n_channels'],
                        n_samples=self.mi_samples_config,
                        fold_id=fold_idx,
                    )
                    actually_deep = is_dl and isinstance(clf, _TorchClassifierWrapper)
                    
                    if actually_deep:
                        clf.fit(X_train_pp_f, y_train_f)
                        probas = clf.predict_proba(X_test_pp_f)
                    else:
                        csp_dva = CSPFeatureExtractor(
                            n_components=self.config['features']['csp_components'],
                            reg=self.config['features']['csp_reg']
                        )
                        X_train_feat_dva = csp_dva.fit_transform(X_train_pp_f, y_train_f)
                        X_test_feat_dva = csp_dva.transform(X_test_pp_f)
                        bp_train_dva = self.band_power.extract(X_train_pp_f)
                        bp_test_dva = self.band_power.extract(X_test_pp_f)
                        clf.fit(np.hstack([X_train_feat_dva, bp_train_dva]), y_train_f)
                        probas = clf.predict_proba(np.hstack([X_test_feat_dva, bp_test_dva]))
                    
                    for i in range(len(y_test_f)):
                        sq = self.preprocessor.compute_signal_quality(X_test_pp_f[i:i+1])
                        dva_result = self.dva.validate(
                            probas[i], true_label=y_test_f[i],
                            signal_quality=sq['signal_quality_score']
                        )
                        all_dva_results.append(dva_result)
                        all_y_test_dva.append(int(y_test_f[i]))
                except Exception as e:
                    logger.warning(f"  DVA fold {fold_idx} error: {e}")
                    continue
            
            if all_dva_results:
                subject_results['dva_results'] = {
                    'statistics': self.dva.get_statistics(),
                    'n_accepted': sum(1 for r in all_dva_results if r['decision'] == 'accept'),
                    'n_rejected': sum(1 for r in all_dva_results if r['decision'] == 'reject'),
                    'n_reviewed': sum(1 for r in all_dva_results if r['decision'] == 'review'),
                    'accepted_accuracy': float(np.mean([
                        r['prediction'] == all_y_test_dva[i]
                        for i, r in enumerate(all_dva_results)
                        if r['decision'] == 'accept'
                    ])) if any(r['decision'] == 'accept' for r in all_dva_results) else 0,
                }
        
        subject_results['apa_stats'] = self.apa.get_statistics()
        return subject_results

    def _run_subject(self, subject_id: int, generator, 
                    n_trials_per_class: int, classifiers: list) -> dict:
        """Run experiment for a single subject."""
        # Generate data
        X, y = generator.generate_subject(subject_id, n_trials_per_class)
        logger.info(f"  Data shape: X={X.shape}, y={y.shape}")
        logger.info(f"  Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")
        
        # Split: session 1 (train) vs session 2 (test) simulation
        n_total = len(y)
        n_train = int(n_total * 0.8)
        
        rng = np.random.RandomState(42 + subject_id)
        indices = rng.permutation(n_total)
        train_idx = indices[:n_train]
        test_idx = indices[n_train:]
        
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
        
        subject_results = {
            'n_train': len(y_train),
            'n_test': len(y_test),
            'classifiers': {},
        }
        
        # ================================================================
        # CONDITION 1: Baseline (standard preprocessing)
        # ================================================================
        logger.info("  Processing: Baseline")
        # Preprocess with different profiles for traditional vs deep models
        X_train_pp_mod = self.preprocessor.process(X_train, 'moderate')
        X_test_pp_mod = self.preprocessor.process(X_test, 'moderate')
        X_train_pp_deep = self.preprocessor.process(X_train, 'deep')
        X_test_pp_deep = self.preprocessor.process(X_test, 'deep')
        
        # CSP + BandPower features (for traditional classifiers)
        csp_train = CSPFeatureExtractor(
            n_components=self.config['features']['csp_components'],
            reg=self.config['features']['csp_reg']
        )
        X_train_feat = csp_train.fit_transform(X_train_pp_mod, y_train)
        X_test_feat = csp_train.transform(X_test_pp_mod)
        bp_train = self.band_power.extract(X_train_pp_mod)
        bp_test = self.band_power.extract(X_test_pp_mod)
        X_train_combined = np.hstack([X_train_feat, bp_train])
        X_test_combined = np.hstack([X_test_feat, bp_test])
        
        for clf_name in classifiers:
            logger.info(f"  Training {clf_name}...")
            is_deep_clf = clf_name in ClassifierFactory.DEEP_CLASSIFIERS
            if is_deep_clf:
                bl_train_data = np.zeros((len(y_train), 1))
                bl_test_data = np.zeros((len(y_test), 1))
                bl_train_raw = X_train_pp_deep
                bl_test_raw = X_test_pp_deep
            else:
                bl_train_data = X_train_combined
                bl_test_data = X_test_combined
                bl_train_raw = X_train_pp_mod
                bl_test_raw = X_test_pp_mod
            try:
                result = self._train_and_evaluate(
                    clf_name, bl_train_data, y_train,
                    bl_test_data, y_test,
                    bl_train_raw, bl_test_raw,
                    condition='baseline'
                )
                subject_results['classifiers'][f'{clf_name}_baseline'] = result
                logger.info(f"    Accuracy: {result['accuracy']:.4f}, Kappa: {result['kappa']:.4f}")
            except Exception as e:
                logger.error(f"    Error with {clf_name}: {e}")
                subject_results['classifiers'][f'{clf_name}_baseline'] = {
                    'accuracy': 0, 'kappa': 0, 'error': str(e)
                }
        
        # ================================================================
        # CONDITION 2: APA-enhanced preprocessing
        # ================================================================
        logger.info("  Processing: APA-enhanced preprocessing")
        
        # A1: Session-level APA for deep models
        # Compute session-level signal quality for session-level APA
        sample_idx = np.random.choice(len(X_train), min(20, len(X_train)), replace=False)
        avg_sq = {}
        for feat in ['snr', 'artifact_ratio', 'line_noise', 'signal_quality_score']:
            vals = [self.preprocessor.compute_signal_quality(X_train[si:si+1])[feat]
                    for si in sample_idx]
            avg_sq[feat] = float(np.mean(vals))
        session_action = self.apa.select_action(avg_sq)
        session_profile = self.apa.get_action_name(session_action)
        
        # Session-level APA-processed data (for deep models)
        X_train_apa_session = self.preprocessor.process(X_train, session_profile)
        X_test_apa_session = self.preprocessor.process(X_test, session_profile)
        
        # Session-level APA feedback
        sq_after_session = {}
        for feat in ['snr', 'artifact_ratio', 'line_noise', 'signal_quality_score']:
            vals = [self.preprocessor.compute_signal_quality(X_train_apa_session[si:si+1])[feat]
                    for si in sample_idx]
            sq_after_session[feat] = float(np.mean(vals))
        reward_session = (sq_after_session['signal_quality_score'] - avg_sq['signal_quality_score']) * 2
        self.apa.update(avg_sq, session_action, reward_session, sq_after_session, done=True)
        
        # Trial-level APA-processed data (for traditional classifiers)
        X_train_apa = np.zeros_like(X_train)
        apa_actions_train = []
        
        for i in range(len(X_train)):
            sq = self.preprocessor.compute_signal_quality(X_train[i:i+1])
            action = self.apa.select_action(sq)
            profile = self.apa.get_action_name(action)
            X_train_apa[i] = self.preprocessor.process(X_train[i:i+1], profile)[0]
            apa_actions_train.append(action)
            
            sq_after = self.preprocessor.compute_signal_quality(X_train_apa[i:i+1])
            reward = (sq_after['signal_quality_score'] - sq['signal_quality_score']) * 2
            self.apa.update(sq, action, reward, sq_after, done=True)
        
        X_test_apa = np.zeros_like(X_test)
        apa_actions_test = []
        for i in range(len(X_test)):
            sq = self.preprocessor.compute_signal_quality(X_test[i:i+1])
            action = self.apa.select_action(sq)
            profile = self.apa.get_action_name(action)
            X_test_apa[i] = self.preprocessor.process(X_test[i:i+1], profile)[0]
            apa_actions_test.append(action)
        
        # CSP + BandPower features (for traditional classifiers only)
        csp_apa = CSPFeatureExtractor(
            n_components=self.config['features']['csp_components'],
            reg=self.config['features']['csp_reg']
        )
        X_train_feat_apa = csp_apa.fit_transform(X_train_apa, y_train)
        X_test_feat_apa = csp_apa.transform(X_test_apa)
        
        bp_train_apa = self.band_power.extract(X_train_apa)
        bp_test_apa = self.band_power.extract(X_test_apa)
        
        X_train_combined_apa = np.hstack([X_train_feat_apa, bp_train_apa])
        X_test_combined_apa = np.hstack([X_test_feat_apa, bp_test_apa])
        
        apa_deep_disabled = not self.config.get('apa', {}).get('deep_model_enabled', False)
        
        for clf_name in classifiers:
            is_deep_clf = clf_name in ClassifierFactory.DEEP_CLASSIFIERS
            
            # Phase 1D: Skip APA for deep models when disabled
            if is_deep_clf and apa_deep_disabled:
                # Copy baseline result
                baseline_key = f'{clf_name}_baseline'
                apa_key = f'{clf_name}_apa'
                if baseline_key in subject_results['classifiers']:
                    subject_results['classifiers'][apa_key] = subject_results['classifiers'][baseline_key].copy()
                    subject_results['classifiers'][apa_key]['condition'] = 'apa (copied from baseline)'
                continue
            
            logger.info(f"  Training {clf_name} (APA)...")
            if is_deep_clf:
                apa_train_data = np.zeros((len(y_train), 1))
                apa_test_data = np.zeros((len(y_test), 1))
                # Use deep profile for deep models even in APA condition
                apa_train_raw = self.preprocessor.process(X_train, 'deep')
                apa_test_raw = self.preprocessor.process(X_test, 'deep')
            else:
                apa_train_data = X_train_combined_apa
                apa_test_data = X_test_combined_apa
                apa_train_raw = X_train_apa
                apa_test_raw = X_test_apa
            try:
                result = self._train_and_evaluate(
                    clf_name, apa_train_data, y_train,
                    apa_test_data, y_test,
                    apa_train_raw, apa_test_raw,
                    condition='apa'
                )
                subject_results['classifiers'][f'{clf_name}_apa'] = result
                logger.info(f"    Accuracy: {result['accuracy']:.4f}, Kappa: {result['kappa']:.4f}")
            except Exception as e:
                logger.error(f"    Error with {clf_name}+APA: {e}")
                subject_results['classifiers'][f'{clf_name}_apa'] = {
                    'accuracy': 0, 'kappa': 0, 'error': str(e)
                }
        
        # ================================================================
        # CONDITION 3: APA + DVA
        # ================================================================
        logger.info("  Processing: APA + DVA validation")
        self.dva.reset()
        
        # Use best baseline classifier for DVA evaluation
        best_clf_name = None
        best_acc = 0
        for clf_name in classifiers:
            key = f'{clf_name}_apa'
            if key in subject_results['classifiers']:
                acc = subject_results['classifiers'][key].get('accuracy', 0)
                if acc > best_acc:
                    best_acc = acc
                    best_clf_name = clf_name
        
        if best_clf_name:
            # Re-train best classifier
            is_dl = best_clf_name in ClassifierFactory.DEEP_CLASSIFIERS
            clf = ClassifierFactory.create(
                best_clf_name,
                self.config['classifiers'].get(best_clf_name, {}),
                n_classes=self.config['dataset']['n_classes'],
                n_channels=self.config['dataset']['n_channels'],
                n_samples=self.mi_samples_config,
            )
            
            if is_dl:
                dva_train_raw = self.preprocessor.process(X_train, 'deep')
                dva_test_raw = self.preprocessor.process(X_test, 'deep')
                clf.fit(dva_train_raw, y_train)
                probas = clf.predict_proba(dva_test_raw)
            else:
                clf.fit(X_train_combined_apa, y_train)
                probas = clf.predict_proba(X_test_combined_apa)
            
            dva_results = []
            for i in range(len(y_test)):
                sq = self.preprocessor.compute_signal_quality(X_test_apa[i:i+1])
                dva_result = self.dva.validate(
                    probas[i], 
                    true_label=y_test[i],
                    signal_quality=sq['signal_quality_score']
                )
                dva_results.append(dva_result)
            
            subject_results['dva_results'] = {
                'statistics': self.dva.get_statistics(),
                'n_accepted': sum(1 for r in dva_results if r['decision'] == 'accept'),
                'n_rejected': sum(1 for r in dva_results if r['decision'] == 'reject'),
                'n_reviewed': sum(1 for r in dva_results if r['decision'] == 'review'),
                'accepted_accuracy': np.mean([
                    r['prediction'] == y_test[i] 
                    for i, r in enumerate(dva_results) 
                    if r['decision'] == 'accept'
                ]) if any(r['decision'] == 'accept' for r in dva_results) else 0,
            }
            
            logger.info(f"  DVA Stats: {self.dva.get_statistics()}")
        
        # Store APA stats for this subject
        subject_results['apa_stats'] = self.apa.get_statistics()
        
        return subject_results
    
    def _train_and_evaluate(self, clf_name: str,
                           X_train: np.ndarray, y_train: np.ndarray,
                           X_test: np.ndarray, y_test: np.ndarray,
                           X_train_raw: Optional[np.ndarray] = None,
                           X_test_raw: Optional[np.ndarray] = None,
                           condition: str = 'baseline',
                           fold_id: int = 0) -> dict:
        """Train classifier and evaluate.
        
        Phase 1B: augmentation is now handled inside _TorchClassifierWrapper.fit()
        so no external augmentation is applied here.
        """
        is_deep = clf_name in ClassifierFactory.DEEP_CLASSIFIERS
        
        clf = ClassifierFactory.create(
            clf_name,
            self.config['classifiers'].get(clf_name, {}),
            n_classes=self.config['dataset']['n_classes'],
            n_channels=self.config['dataset']['n_channels'],
            n_samples=self.mi_samples_config,
            fold_id=fold_id,
        )
        
        actually_deep = is_deep and isinstance(clf, _TorchClassifierWrapper)
        
        start_time = time.time()
        
        if actually_deep and X_train_raw is not None:
            # Deep models: train on raw preprocessed EEG (augmentation inside fit())
            clf.fit(X_train_raw, y_train)
            y_pred = clf.predict(X_test_raw)
            y_proba = clf.predict_proba(X_test_raw)
        else:
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            y_proba = clf.predict_proba(X_test)
        
        train_time = time.time() - start_time
        
        metrics = EvaluationMetrics.compute_all(
            y_test, y_pred, y_proba,
            class_names=self.config['dataset']['class_names']
        )
        metrics['train_time_sec'] = round(train_time, 2)
        metrics['classifier'] = clf_name
        metrics['condition'] = condition
        
        return metrics
    
    def _compute_summary(self, classifiers: list):
        """Compute summary statistics across subjects."""
        summary = {}
        
        for clf_name in classifiers:
            for condition in ['baseline', 'apa']:
                key = f'{clf_name}_{condition}'
                accs = []
                kappas = []
                
                for subj, subj_data in self.results['subjects'].items():
                    clf_data = subj_data.get('classifiers', {}).get(key, {})
                    if 'accuracy' in clf_data and clf_data.get('accuracy', 0) > 0:
                        accs.append(clf_data['accuracy'])
                        kappas.append(clf_data['kappa'])
                
                if accs:
                    summary[key] = {
                        'accuracy_mean': float(np.mean(accs)),
                        'accuracy_std': float(np.std(accs)),
                        'kappa_mean': float(np.mean(kappas)),
                        'kappa_std': float(np.std(kappas)),
                        'accuracy_min': float(np.min(accs)),
                        'accuracy_max': float(np.max(accs)),
                        'n_subjects': len(accs),
                    }
        
        self.results['summary'] = summary
    
    def _compute_statistical_tests(self, classifiers: list):
        """Compute statistical tests (Wilcoxon, McNemar)."""
        tests = {}
        
        for clf_name in classifiers:
            baseline_accs = []
            apa_accs = []
            
            for subj, subj_data in self.results['subjects'].items():
                b = subj_data.get('classifiers', {}).get(f'{clf_name}_baseline', {})
                a = subj_data.get('classifiers', {}).get(f'{clf_name}_apa', {})
                
                if b.get('accuracy', 0) > 0 and a.get('accuracy', 0) > 0:
                    baseline_accs.append(b['accuracy'])
                    apa_accs.append(a['accuracy'])
            
            if len(baseline_accs) >= 3:
                tests[f'{clf_name}_baseline_vs_apa'] = EvaluationMetrics.wilcoxon_test(
                    baseline_accs, apa_accs
                )
                tests[f'{clf_name}_baseline_vs_apa']['baseline_mean'] = np.mean(baseline_accs)
                tests[f'{clf_name}_baseline_vs_apa']['apa_mean'] = np.mean(apa_accs)
                tests[f'{clf_name}_baseline_vs_apa']['improvement'] = np.mean(apa_accs) - np.mean(baseline_accs)
        
        self.results['statistical_tests'] = tests
    
    def _save_results(self):
        """Save results to files."""
        # JSON
        results_path = self.output_dir / 'experiment_results.json'
        
        # Convert numpy types for JSON serialization
        def convert(obj):
            if isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, np.bool_):
                return bool(obj)
            return obj
        
        with open(results_path, 'w') as f:
            json.dump(self.results, f, indent=2, default=convert)
        
        logger.info(f"Results saved to {results_path}")
        
        # CSV summary
        csv_path = self.output_dir / 'summary_table.csv'
        with open(csv_path, 'w') as f:
            f.write("Classifier,Condition,Accuracy_Mean,Accuracy_Std,Kappa_Mean,Kappa_Std,N_Subjects\n")
            for key, stats in self.results.get('summary', {}).items():
                parts = key.rsplit('_', 1)
                clf_name = parts[0] if len(parts) > 1 else key
                condition = parts[-1] if len(parts) > 1 else 'unknown'
                f.write(
                    f"{clf_name},{condition},"
                    f"{stats['accuracy_mean']:.4f},{stats['accuracy_std']:.4f},"
                    f"{stats['kappa_mean']:.4f},{stats['kappa_std']:.4f},"
                    f"{stats['n_subjects']}\n"
                )
        logger.info(f"Summary CSV saved to {csv_path}")
    
    def _print_summary(self):
        """Print formatted summary."""
        print("\n" + "=" * 80)
        print("EXPERIMENT RESULTS SUMMARY")
        print("=" * 80)
        
        print(f"\n{'Classifier':<25} {'Condition':<12} {'Accuracy':>10} {'Kappa':>10}")
        print("-" * 60)
        
        for key, stats in sorted(self.results.get('summary', {}).items()):
            parts = key.rsplit('_', 1)
            clf_name = parts[0] if len(parts) > 1 else key
            condition = parts[-1]
            print(
                f"{clf_name:<25} {condition:<12} "
                f"{stats['accuracy_mean']:.4f}+/-{stats['accuracy_std']:.4f} "
                f"{stats['kappa_mean']:.4f}+/-{stats['kappa_std']:.4f}"
            )
        
        print("\n" + "-" * 60)
        print("STATISTICAL TESTS (Baseline vs APA)")
        print("-" * 60)
        
        for key, test in self.results.get('statistical_tests', {}).items():
            print(f"\n{key}:")
            print(f"  Baseline: {test.get('baseline_mean', 0):.4f}")
            print(f"  APA:      {test.get('apa_mean', 0):.4f}")
            print(f"  Improvement: {test.get('improvement', 0):+.4f}")
            print(f"  p-value: {test.get('p_value', 1):.4f}")
            sig = "YES" if test.get('significant_005', False) else "NO"
            print(f"  Significant (p<0.05): {sig}")
        
        print("\n" + "=" * 80)
        
        # APA stats
        apa_stats = self.apa.get_statistics()
        print("\nAPA Agent Statistics:")
        print(f"  Epsilon: {apa_stats['epsilon']:.4f}")
        print(f"  Q-table coverage: {apa_stats['q_table_coverage']:.2%}")
        print(f"  Action distribution: {apa_stats['action_distribution']}")
        
        print("\n" + "=" * 80)


# ============================================================================
# MAIN
# ============================================================================


print('Framework loaded: all classes available.')

## 4. Data Exploration <a name="sec4"></a>

In [ ]:
# ==============================================================================
# Data Exploration
# ==============================================================================

if USE_REAL_DATA:
    # --- Load real data for exploration ---
    real_loader = RealBCI2aLoader(DATA_DIR, EXPERIMENT_CONFIG)
    X_demo, y_demo = real_loader.load_subject(1, training=True, mi_period_only=True)
    data_info = real_loader.get_dataset_info()
    data_label = "Real BCI IV-2a (Subject 1)"
    fs_demo = 250
    ch_names = real_loader.channel_names
    print(f"Real data: X={X_demo.shape}, y={y_demo.shape}")
    print(f"Classes: {dict(zip(*np.unique(y_demo, return_counts=True)))}")
else:
    # --- Synthetic fallback ---
    gen = SyntheticBCI2aGenerator(EXPERIMENT_CONFIG)
    X_demo, y_demo = gen.generate_subject(1, n_trials_per_class=18)
    data_label = "Synthetic Data (Subject 1)"
    fs_demo = 250
    ch_names = gen.channel_names
    print(f"Synthetic data: X={X_demo.shape}, y={y_demo.shape}")
    print(f"Classes: {dict(zip(*np.unique(y_demo, return_counts=True)))}")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# (a) Raw EEG
trial = X_demo[0]; t = np.arange(trial.shape[1]) / fs_demo
for i, (name, idx) in enumerate([('C3',7),('C4',11),('Cz',9),('Fz',0)]):
    axes[0,0].plot(t, trial[idx] + i*50, label=name, linewidth=0.8)
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_ylabel('Amplitude (a.u.)')
axes[0,0].set_title(f'(a) Raw EEG Trial -- {data_label}'); axes[0,0].legend()

# (b) PSD by class
for ci, cn in enumerate(EXPERIMENT_CONFIG['dataset']['class_names']):
    ct = X_demo[y_demo == ci]
    if len(ct) == 0:
        continue
    psds = [scipy_signal.welch(tr[7], fs=fs_demo, nperseg=min(256, tr.shape[1]))[1] for tr in ct[:10]]
    f_ax = scipy_signal.welch(ct[0,7], fs=fs_demo, nperseg=min(256, ct[0].shape[1]))[0]
    axes[0,1].semilogy(f_ax, np.mean(psds, axis=0), label=cn, linewidth=1.5)
axes[0,1].set_xlim(0, 50); axes[0,1].set_xlabel('Frequency (Hz)')
axes[0,1].set_ylabel('PSD'); axes[0,1].set_title(f'(b) PSD at C3 by Class')
axes[0,1].legend(); axes[0,1].axvspan(8,12,alpha=0.15,color='red')
axes[0,1].axvspan(12,30,alpha=0.1,color='blue')

# (c) SNR distribution
pp = EEGPreprocessor(EXPERIMENT_CONFIG)
sqs = [pp.compute_signal_quality(X_demo[i:i+1]) for i in range(min(len(X_demo), 100))]
snrs = [s['snr'] for s in sqs]
axes[1,0].hist(snrs, bins=20, color='#4C72B0', alpha=0.7, edgecolor='white')
axes[1,0].set_xlabel('SNR (dB)'); axes[1,0].set_ylabel('Count')
axes[1,0].set_title('(c) SNR Distribution')
axes[1,0].axvline(np.mean(snrs), color='red', ls='--', label=f'Mean={np.mean(snrs):.1f}dB')
axes[1,0].legend()

# (d) Channel variance
ch_vars = np.var(trial, axis=1)
axes[1,1].barh(ch_names, ch_vars, color='#55A868', alpha=0.8)
axes[1,1].set_xlabel('Variance'); axes[1,1].set_title('(d) Channel Variance (Trial 1)')
axes[1,1].invert_yaxis()

data_type_tag = "REAL" if USE_REAL_DATA else "SYNTHETIC"
plt.suptitle(f'Figure 1: Data Exploration [{data_type_tag}]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig1_data_exploration.png'))
plt.show()


In [ ]:
# ==============================================================================
# Figure 12: Preprocessing Profile Comparison (Moderate vs Deep)
# ==============================================================================
# Generate comparison using synthetic or real data
if USE_REAL_DATA:
    _loader = RealBCI2aLoader(DATA_DIR, EXPERIMENT_CONFIG)
    _X_demo, _y_demo = _loader.load_subject(1, training=True, mi_period_only=True)
else:
    _gen = SyntheticBCI2aGenerator(EXPERIMENT_CONFIG)
    _X_demo, _y_demo = _gen.generate_subject(1, n_trials_per_class=18)

_pp = EEGPreprocessor(EXPERIMENT_CONFIG)
_trial = _X_demo[0:1]  # First trial
_t_axis = np.arange(_trial.shape[2]) / 250

_pp_moderate = _pp.process(_trial, 'moderate')[0]
_pp_deep = _pp.process(_trial, 'deep')[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Time series at C3
ch_idx = 7  # C3
axes[0].plot(_t_axis, _pp_moderate[ch_idx], label='Moderate (8-30Hz, z-norm)', alpha=0.8)
axes[0].plot(_t_axis, _pp_deep[ch_idx], label='Deep (4-40Hz, no norm)', alpha=0.8)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')
axes[0].set_title('(a) C3 Time Series: Moderate vs Deep')
axes[0].legend()

# (b) PSD comparison
from scipy.signal import welch as _welch
_f_mod, _psd_mod = _welch(_pp_moderate[ch_idx], fs=250, nperseg=min(256, _pp_moderate.shape[1]))
_f_deep, _psd_deep = _welch(_pp_deep[ch_idx], fs=250, nperseg=min(256, _pp_deep.shape[1]))
axes[1].semilogy(_f_mod, _psd_mod, label='Moderate', alpha=0.8)
axes[1].semilogy(_f_deep, _psd_deep, label='Deep', alpha=0.8)
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('PSD')
axes[1].set_title('(b) PSD: Moderate vs Deep')
axes[1].set_xlim(0, 50)
axes[1].axvspan(8, 12, alpha=0.1, color='red', label='mu band')
axes[1].axvspan(12, 30, alpha=0.1, color='blue', label='beta band')
axes[1].legend()

plt.suptitle('Figure 12: Preprocessing Profile Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig12_preproc_compare.png'))
plt.show()


## 5. Run Full Experiment <a name="sec5"></a>

In [ ]:
# ==============================================================================
# Run Full Experiment (9 Subjects x Classifiers x 3 Conditions)
# ==============================================================================
# Uses REAL data when available, falls back to SYNTHETIC otherwise.

N_SUBJECTS = 9
N_PER_CLASS = 72
CLASSIFIERS_LIST = [
    'lda', 'svm',                                              # Traditional ML
    'eegnet', 'shallow_convnet', 'deep_convnet',               # Base DL
    'atcnet', 'eegconformer', 'eegtcnet', 'ctnet',            # Braindecode
    'mscformer',                                                # MSCFormer (Zhao et al., 2025)
]

runner = ExperimentRunner(config=EXPERIMENT_CONFIG, output_dir=RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results')

t0 = time.time()

if USE_REAL_DATA:
    print("=" * 70)
    print("RUNNING WITH REAL BCI IV-2a DATA")
    print("=" * 70)
    results = runner.run_real_experiment(
        data_dir=DATA_DIR,
        classifiers=CLASSIFIERS_LIST,
        n_folds=5
    )
else:
    print("=" * 70)
    print("WARNING: Running with SYNTHETIC data (no real .mat files found)")
    print("Results are for demonstration/debugging only.")
    print("=" * 70)
    results = runner.run_synthetic_experiment(
        n_subjects=N_SUBJECTS,
        n_trials_per_class=N_PER_CLASS,
        classifiers=CLASSIFIERS_LIST
    )

elapsed = time.time() - t0
print(f"\nTotal time: {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Data source: {'REAL BCI IV-2a' if USE_REAL_DATA else 'SYNTHETIC'}")


## 6. Results & Visualization <a name="sec6"></a>

In [ ]:
# ==============================================================================
# TABLE 1: Classification Results Summary
# ==============================================================================
rows = []
for key, s in sorted(results.get('summary', {}).items()):
    parts = key.rsplit('_', 1)
    rows.append({
        'Classifier': parts[0].upper(), 'Condition': parts[1].upper(),
        'Accuracy': f"{s['accuracy_mean']:.4f} +/- {s['accuracy_std']:.4f}",
        'Kappa': f"{s['kappa_mean']:.4f} +/- {s['kappa_std']:.4f}",
        'Min': f"{s['accuracy_min']:.4f}", 'Max': f"{s['accuracy_max']:.4f}",
        'N': s['n_subjects'], '_acc': s['accuracy_mean'],
    })
df_summary = pd.DataFrame(rows)
print("=" * 90)
print("TABLE 1: Classification Results (Mean +/- Std across subjects)")
print("=" * 90)
print(df_summary[['Classifier','Condition','Accuracy','Kappa','Min','Max','N']].to_string(index=False))
print("=" * 90)


In [ ]:
# ==============================================================================
# TABLE 2: Per-Subject Accuracy (%)
# ==============================================================================
rows = []
for sid, sd in sorted(results['subjects'].items()):
    row = {'Subject': sid}
    for k, v in sd.get('classifiers', {}).items():
        row[k] = round(v.get('accuracy', 0) * 100, 2)
    rows.append(row)
df_per_subj = pd.DataFrame(rows).set_index('Subject')
print("\nTABLE 2: Per-Subject Accuracy (%)")
print("=" * 120)
print(df_per_subj.to_string())
print("=" * 120)


In [ ]:
# ==============================================================================
# Figure 2: Accuracy Comparison Bar Chart
# ==============================================================================
summary = results['summary']
clf_names = sorted(set(k.rsplit('_',1)[0] for k in summary))
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(clf_names)); w = 0.35
ba = [summary.get(f'{c}_baseline',{}).get('accuracy_mean',0) for c in clf_names]
bs = [summary.get(f'{c}_baseline',{}).get('accuracy_std',0) for c in clf_names]
aa = [summary.get(f'{c}_apa',{}).get('accuracy_mean',0) for c in clf_names]
ast_ = [summary.get(f'{c}_apa',{}).get('accuracy_std',0) for c in clf_names]
axes[0].bar(x-w/2, ba, w, yerr=bs, label='Baseline', color='#4C72B0', capsize=4, alpha=0.85)
axes[0].bar(x+w/2, aa, w, yerr=ast_, label='APA-Enhanced', color='#DD8452', capsize=4, alpha=0.85)
axes[0].set_xlabel('Classifier'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('(a) Accuracy'); axes[0].set_xticks(x)
axes[0].set_xticklabels([c.upper() for c in clf_names], rotation=20)
axes[0].legend(); axes[0].set_ylim(0, 1.0); axes[0].axhline(0.25, color='gray', ls='--', alpha=0.5)
import matplotlib.ticker as mtick
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
bk = [summary.get(f'{c}_baseline',{}).get('kappa_mean',0) for c in clf_names]
bks = [summary.get(f'{c}_baseline',{}).get('kappa_std',0) for c in clf_names]
ak = [summary.get(f'{c}_apa',{}).get('kappa_mean',0) for c in clf_names]
aks = [summary.get(f'{c}_apa',{}).get('kappa_std',0) for c in clf_names]
axes[1].bar(x-w/2, bk, w, yerr=bks, label='Baseline', color='#4C72B0', capsize=4, alpha=0.85)
axes[1].bar(x+w/2, ak, w, yerr=aks, label='APA-Enhanced', color='#DD8452', capsize=4, alpha=0.85)
axes[1].set_xlabel('Classifier'); axes[1].set_ylabel("Cohen's Kappa")
axes[1].set_title("(b) Kappa"); axes[1].set_xticks(x)
axes[1].set_xticklabels([c.upper() for c in clf_names], rotation=20); axes[1].legend()
plt.suptitle('Figure 2: Baseline vs APA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig2_accuracy.png'))
plt.show()


In [ ]:
# ==============================================================================
# Figure 3: Per-Subject Accuracy Heatmap
# ==============================================================================
subjects = sorted(results['subjects'].keys())
all_conds = sorted(set(k for sd in results['subjects'].values() for k in sd.get('classifiers',{}).keys()))
mat = np.zeros((len(subjects), len(all_conds)))
for i, s in enumerate(subjects):
    for j, c in enumerate(all_conds):
        mat[i,j] = results['subjects'][s].get('classifiers',{}).get(c,{}).get('accuracy',0)*100
fig, ax = plt.subplots(figsize=(max(14, len(all_conds)*1.2), max(6, len(subjects)*0.7)))
sns.heatmap(mat, annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=[c.replace('_','\n') for c in all_conds],
            yticklabels=subjects, ax=ax, vmin=0, vmax=100,
            cbar_kws={'label': 'Accuracy (%)'})
ax.set_title('Figure 3: Per-Subject Accuracy Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig3_heatmap.png'))
plt.show()


In [ ]:
# ==============================================================================
# Figure 4: Confusion Matrices (multiple models)
# ==============================================================================
class_names = EXPERIMENT_CONFIG['dataset']['class_names']
summary_items = [(k, v) for k, v in results.get('summary', {}).items() if k.endswith('_baseline')]
summary_items.sort(key=lambda x: -x[1].get('accuracy_mean', 0))
show_models = []
for k, v in summary_items:
    clf_name = k.rsplit('_', 1)[0]
    if len(show_models) < 3 and clf_name not in show_models:
        show_models.append(clf_name)
if not show_models:
    show_models = ['mscformer']

n_cols = len(show_models)
fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 6))
if n_cols == 1:
    axes = [axes]
for ai, clf_name in enumerate(show_models):
    key = f'{clf_name}_baseline'
    cm_agg = np.zeros((4, 4))
    for sd in results['subjects'].values():
        cm = sd.get('classifiers',{}).get(key,{}).get('confusion_matrix')
        if cm: cm_agg += np.array(cm)
    cm_norm = cm_agg / (cm_agg.sum(axis=1, keepdims=True) + 1e-10) * 100
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[ai], vmin=0, vmax=100)
    acc = results['summary'].get(key,{}).get('accuracy_mean', 0)
    axes[ai].set_title(f'({chr(97+ai)}) {clf_name.upper()} ({acc:.2%})')
    axes[ai].set_xlabel('Predicted'); axes[ai].set_ylabel('True')
plt.suptitle('Figure 4: Confusion Matrices (Best Models)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig4_confusion.png'))
plt.show()


In [ ]:
# ==============================================================================
# Figure 5: APA Agent Analysis
# ==============================================================================
apa_stats = runner.apa.get_statistics()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Action distribution
ad = apa_stats['action_distribution']
c_colors = ['#55A868', '#4C72B0', '#C44E52']
bars = axes[0].bar(ad.keys(), ad.values(), color=c_colors, alpha=0.85)
axes[0].set_title('(a) Action Distribution'); axes[0].set_ylabel('Count')
for b in bars:
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{int(b.get_height())}',
                 ha='center', va='bottom', fontsize=9)

# (b) Q-table
qt = runner.apa.q_table
visited = np.any(qt != 0, axis=1)
vq = qt[visited]
if len(vq) > 0:
    n_show = min(20, len(vq))
    sns.heatmap(vq[:n_show], annot=True, fmt='.2f', cmap='RdYlGn',
                xticklabels=['Conservative','Moderate','Aggressive'], ax=axes[1], center=0)
    axes[1].set_title(f'(b) Q-Table ({n_show} of {runner.apa.n_states} states visited)')

# (c) Learning curve
if runner.apa.episode_rewards:
    rw = runner.apa.episode_rewards
    win = max(1, len(rw) // 20)
    sm = np.convolve(rw, np.ones(win)/win, mode='valid')
    axes[2].plot(sm, color='#4C72B0', alpha=0.8)
    axes[2].set_title('(c) Learning Curve'); axes[2].set_xlabel('Episode'); axes[2].set_ylabel('Reward')

plt.suptitle('Figure 5: APA Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig5_apa.png'))
plt.show()
print(f"\nAPA: epsilon={apa_stats['epsilon']:.4f}, coverage={apa_stats['q_table_coverage']:.2%}, "
      f"episodes={apa_stats['n_episodes']}, avg_reward={apa_stats['avg_reward']:.4f}")


In [ ]:
# ==============================================================================
# Figure 6: DVA Analysis
# ==============================================================================
dva_data = []
for sid, sd in sorted(results['subjects'].items()):
    dr = sd.get('dva_results', {})
    if dr:
        dva_data.append({'Subject': sid, 'Accepted': dr.get('n_accepted',0),
                         'Rejected': dr.get('n_rejected',0), 'Reviewed': dr.get('n_reviewed',0),
                         'Acc_Accepted': dr.get('accepted_accuracy',0)})
if dva_data:
    df_dva = pd.DataFrame(dva_data)
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    subs = df_dva['Subject'].values; x = np.arange(len(subs)); w = 0.25
    axes[0].bar(x-w, df_dva['Accepted'], w, label='Accept', color='#55A868')
    axes[0].bar(x, df_dva['Rejected'], w, label='Reject', color='#C44E52')
    axes[0].bar(x+w, df_dva['Reviewed'], w, label='Review', color='#4C72B0')
    axes[0].set_xticks(x); axes[0].set_xticklabels(subs, rotation=45)
    axes[0].set_title('(a) DVA Decisions per Subject'); axes[0].legend()
    axes[1].bar(subs, df_dva['Acc_Accepted']*100, color='#55A868', alpha=0.8)
    axes[1].set_title('(b) Accuracy of Accepted Classifications')
    axes[1].set_ylabel('Accuracy (%)'); axes[1].axhline(25, color='gray', ls='--')
    axes[1].set_xticklabels(subs, rotation=45)
    # (c) DVA acceptance rate vs overall accuracy
    total_decisions = df_dva['Accepted'] + df_dva['Rejected'] + df_dva['Reviewed']
    accept_rate = (df_dva['Accepted'] / total_decisions.replace(0, 1)) * 100
    axes[2].bar(x - w/2, accept_rate, w, label='Accept Rate (%)', color='#4C72B0', alpha=0.8)
    axes[2].bar(x + w/2, df_dva['Acc_Accepted']*100, w, label='Accepted Acc (%)', color='#55A868', alpha=0.8)
    axes[2].set_xticks(x); axes[2].set_xticklabels(subs, rotation=45)
    axes[2].set_title('(c) Accept Rate vs Accepted Accuracy'); axes[2].legend()
    axes[2].set_ylabel('Percentage (%)')
    plt.suptitle('Figure 6: DVA Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig6_dva.png'))
    plt.show()
    print("\nTABLE 3: DVA Statistics")
    print(df_dva.to_string(index=False))
else:
    print("No DVA results.")


In [ ]:
# ==============================================================================
# Figure 9: Per-Class Accuracy by Model
# ==============================================================================
class_names = EXPERIMENT_CONFIG['dataset']['class_names']
model_names = []
per_class_accs = {cn: [] for cn in class_names}

for clf_name in CLASSIFIERS_LIST:
    key = f'{clf_name}_baseline'
    if key in results.get('summary', {}):
        # Aggregate confusion matrices across subjects
        cm_agg = np.zeros((4, 4))
        for sd in results['subjects'].values():
            cm = sd.get('classifiers', {}).get(key, {}).get('confusion_matrix')
            if cm:
                cm_agg += np.array(cm)
        # Per-class accuracy from diagonal
        row_sums = cm_agg.sum(axis=1)
        for ci, cn in enumerate(class_names):
            if row_sums[ci] > 0:
                per_class_accs[cn].append(cm_agg[ci, ci] / row_sums[ci] * 100)
            else:
                per_class_accs[cn].append(0)
        model_names.append(clf_name.upper())

if model_names:
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(class_names))
    n_models = len(model_names)
    w = 0.8 / n_models
    colors = plt.cm.tab10(np.linspace(0, 1, n_models))

    for mi, mn in enumerate(model_names):
        vals = [per_class_accs[cn][mi] for cn in class_names]
        ax.bar(x + mi * w - (n_models - 1) * w / 2, vals, w, label=mn, color=colors[mi], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Figure 9: Per-Class Accuracy by Model', fontsize=14, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax.axhline(25, color='gray', ls='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig9_per_class.png'))
    plt.show()


In [ ]:
# ==============================================================================
# Figure 11: Subject Difficulty Ranking
# ==============================================================================
subjects = sorted(results.get('subjects', {}).keys())
if subjects:
    # Compute average accuracy per subject across all classifiers
    subj_avg = {}
    for sid in subjects:
        accs = []
        for k, v in results['subjects'][sid].get('classifiers', {}).items():
            if 'accuracy' in v and v['accuracy'] > 0:
                accs.append(v['accuracy'] * 100)
        subj_avg[sid] = np.mean(accs) if accs else 0

    # Sort by average accuracy
    sorted_subjs = sorted(subjects, key=lambda s: subj_avg[s])
    
    fig, ax = plt.subplots(figsize=(14, 6))
    colors_line = plt.cm.tab10(np.linspace(0, 1, len(CLASSIFIERS_LIST)))
    
    for ci, clf_name in enumerate(CLASSIFIERS_LIST):
        key = f'{clf_name}_baseline'
        vals = []
        for sid in sorted_subjs:
            acc = results['subjects'][sid].get('classifiers', {}).get(key, {}).get('accuracy', 0) * 100
            vals.append(acc)
        ax.plot(range(len(sorted_subjs)), vals, 'o-', label=clf_name.upper(),
                color=colors_line[ci], alpha=0.7, markersize=5)
    
    ax.set_xticks(range(len(sorted_subjs)))
    ax.set_xticklabels(sorted_subjs, rotation=45)
    ax.set_xlabel('Subject (sorted by avg accuracy, easiest right)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Figure 11: Subject Difficulty Ranking', fontsize=14, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax.axhline(25, color='gray', ls='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig11_difficulty.png'))
    plt.show()


In [ ]:
# ==============================================================================
# TABLE 7: Per-Class Precision, Recall, F1 by Model
# ==============================================================================
class_names = EXPERIMENT_CONFIG['dataset']['class_names']
table7_rows = []
for clf_name in CLASSIFIERS_LIST:
    key = f'{clf_name}_baseline'
    # Get per-class metrics from last subject or aggregate
    for sid, sd in results.get('subjects', {}).items():
        pc = sd.get('classifiers', {}).get(key, {}).get('per_class', {})
        if pc:
            for cn in class_names:
                if cn in pc:
                    table7_rows.append({
                        'Model': clf_name.upper(),
                        'Subject': sid,
                        'Class': cn,
                        'Precision': f"{pc[cn].get('precision', 0):.3f}",
                        'Recall': f"{pc[cn].get('recall', 0):.3f}",
                        'F1': f"{pc[cn].get('f1', 0):.3f}",
                    })

if table7_rows:
    df_t7 = pd.DataFrame(table7_rows)
    # Show aggregated (mean across subjects) per model per class
    agg_rows = []
    for clf_name in CLASSIFIERS_LIST:
        for cn in class_names:
            mask = (df_t7['Model'] == clf_name.upper()) & (df_t7['Class'] == cn)
            subset = df_t7[mask]
            if len(subset) > 0:
                agg_rows.append({
                    'Model': clf_name.upper(),
                    'Class': cn,
                    'Precision': f"{subset['Precision'].astype(float).mean():.3f}",
                    'Recall': f"{subset['Recall'].astype(float).mean():.3f}",
                    'F1': f"{subset['F1'].astype(float).mean():.3f}",
                })
    if agg_rows:
        df_t7_agg = pd.DataFrame(agg_rows)
        print("=" * 80)
        print("TABLE 7: Per-Class Precision, Recall, F1 (Mean Across Subjects)")
        print("=" * 80)
        print(df_t7_agg.to_string(index=False))
        df_t7_agg.to_csv(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'table7_per_class_metrics.csv'), index=False)
else:
    print("No per-class metrics available.")


## 7. Statistical Tests <a name="sec7"></a>

In [ ]:
# ==============================================================================
# TABLE 4: Statistical Tests
# ==============================================================================
print("=" * 100)
print("TABLE 4: Statistical Tests -- Baseline vs APA")
print("=" * 100)
stat_rows = []
for key, t in sorted(results.get('statistical_tests', {}).items()):
    stat_rows.append({
        'Test': key.replace('_', ' ').title(),
        'Baseline': f"{t.get('baseline_mean',0):.4f}",
        'APA': f"{t.get('apa_mean',0):.4f}",
        'Delta': f"{t.get('improvement',0):+.4f}" if 'improvement' in t else 'N/A',
        'p-value': f"{t.get('p_value',1):.4f}",
        'Sig': 'YES' if t.get('significant_005', False) else 'No',
    })
if stat_rows:
    print(pd.DataFrame(stat_rows).to_string(index=False))
else:
    print("Need >= 3 subjects for tests.")
print("=" * 100)


## 8. Literature Comparison & Differentiation <a name="sec8"></a>

In [ ]:
# ==============================================================================
# TABLE 5: Literature Comparison on BCI IV-2a
# ==============================================================================
data_source_tag = results.get('data_source', 'synthetic')
is_real = data_source_tag == 'real_bci_iv_2a'

lit = [
    ['AMEEGNet','2025','Multi-scale EEGNet + ECA','81.17','0.75','89.83','Session-wise'],
    ['EEG-DCNet','2024','Dilated CNN + SE Attention','83.31','0.78','N/A','Sliding window'],
    ['CIACNet','2025','Dual-branch CNN+CBAM+TCN','85.15','0.80','90.05','Session-wise'],
    ['CLTNet','2025','CNN+LSTM+Transformer','83.02','0.77','87.11','Session-wise'],
    ['EEGEncoder','2025','TCN+Transformer (DSTS)','86.46','0.82','N/A','Session-wise'],
    ['MSCFormer','2025','Multi-scale Conv+Transformer','82.95','0.77','88.00','5-fold CV'],
    ['BrainGridNet','2025','Two-branch Depthwise CNN','80.26','0.75','N/A','10-fold CV'],
    ['ATCNet','2022','Attention TCN','85.38','0.80','N/A','Session-wise'],
]
# Detect eval protocol from results
has_session_wise = False
for sid, sd in results.get('subjects', {}).items():
    if sd.get('eval_protocol') == 'session-wise':
        has_session_wise = True
        break
if is_real:
    eval_protocol = 'Session-wise' if has_session_wise else '5-fold CV'
else:
    eval_protocol = '80/20 split (synthetic)'
for cn in CLASSIFIERS_LIST:
    for cond in ['baseline', 'apa']:
        k = f'{cn}_{cond}'
        if k in results.get('summary', {}):
            s = results['summary'][k]
            source_note = 'Real' if is_real else 'Synth'
            feat_tag = '+CSP' if cn not in ClassifierFactory.DEEP_CLASSIFIERS else ''
            lit.append([f'LLM-EEG ({cn.upper()}) [{source_note}]','2025',
                       f'{cn.upper()}{feat_tag}+{"APA+DVA" if cond=="apa" else "Std"}',
                       f"{s['accuracy_mean']*100:.2f}", f"{s['kappa_mean']:.2f}",
                       'N/A', eval_protocol])
df_lit = pd.DataFrame(lit, columns=['Method','Year','Architecture','Acc_2a(%)','Kappa','Acc_2b(%)','Eval'])
print("=" * 110)
print("TABLE 5: Literature Comparison")
print("=" * 110)
print(df_lit.to_string(index=False))
print("=" * 110)
if not is_real:
    print("\n*** WARNING: LLM-EEG results use SYNTHETIC data. Real BCI IV-2a evaluation pending. ***")
else:
    print(f"\nLLM-EEG results evaluated on REAL BCI Competition IV-2a data ({eval_protocol}).")


In [ ]:
# ==============================================================================
# Figure 8: Accuracy Gap Analysis -- Published vs LLM-EEG
# ==============================================================================
published = {
    'EEGNet': 75.0, 'ATCNet': 85.38, 'EEGConformer': 82.0,
    'EEGTCNet': 78.0, 'CTNet': 80.0, 'MSCFormer': 82.95,
}
llm_eeg_accs = {}
for clf_name in ['eegnet', 'atcnet', 'eegconformer', 'eegtcnet', 'ctnet', 'mscformer']:
    key = f'{clf_name}_baseline'
    if key in results.get('summary', {}):
        llm_eeg_accs[clf_name.upper() if clf_name != 'mscformer' else 'MSCFormer'] = results['summary'][key]['accuracy_mean'] * 100

models = sorted(set(published.keys()) & set(llm_eeg_accs.keys()))
if models:
    fig, ax = plt.subplots(figsize=(12, 6))
    y_pos = np.arange(len(models))
    pub_vals = [published[m] for m in models]
    llm_vals = [llm_eeg_accs[m] for m in models]
    gaps = [p - l for p, l in zip(pub_vals, llm_vals)]

    bars_pub = ax.barh(y_pos + 0.15, pub_vals, 0.3, label='Published', color='#55A868', alpha=0.85)
    bars_llm = ax.barh(y_pos - 0.15, llm_vals, 0.3, label='LLM-EEG', color='#C44E52', alpha=0.85)

    for i, (p, l, g) in enumerate(zip(pub_vals, llm_vals, gaps)):
        ax.annotate(f'{g:.1f}pp gap', xy=(max(p, l) + 1, i), fontsize=9, va='center', color='#333')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(models)
    ax.set_xlabel('Accuracy (%)')
    ax.set_title('Figure 8: Accuracy Gap Analysis (Published vs LLM-EEG)', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.set_xlim(0, 100)
    ax.axvline(25, color='gray', ls='--', alpha=0.3, label='Chance')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig8_accuracy_gap.png'))
    plt.show()
else:
    print("No matching models for gap analysis.")


In [ ]:
# ==============================================================================
# TABLE 9: Computational Cost Comparison
# ==============================================================================
cost_rows = []
for clf_name in CLASSIFIERS_LIST:
    key = f'{clf_name}_baseline'
    times = []
    for sd in results.get('subjects', {}).values():
        t = sd.get('classifiers', {}).get(key, {}).get('train_time_sec', 0)
        if t > 0:
            times.append(t)
    
    # Count parameters for deep models
    n_params = 'N/A'
    is_deep = clf_name in ClassifierFactory.DEEP_CLASSIFIERS
    if is_deep:
        try:
            clf_temp = ClassifierFactory.create(
                clf_name,
                EXPERIMENT_CONFIG['classifiers'].get(clf_name, {}),
                n_classes=4, n_channels=22,
                n_samples=int(EXPERIMENT_CONFIG['dataset']['sampling_rate'] * EXPERIMENT_CONFIG['dataset']['trial_duration']),
            )
            if isinstance(clf_temp, _TorchClassifierWrapper):
                n_params = sum(p.numel() for p in clf_temp.model.parameters())
        except Exception:
            pass
    
    cost_rows.append({
        'Model': clf_name.upper(),
        'Type': 'Deep' if is_deep else 'Traditional',
        'Params': f"{n_params:,}" if isinstance(n_params, int) else n_params,
        'Avg Train Time (s)': f"{np.mean(times):.1f}" if times else 'N/A',
        'Batch Size': EXPERIMENT_CONFIG['classifiers'].get(clf_name, {}).get('batch_size', 'N/A'),
        'Epochs': EXPERIMENT_CONFIG['classifiers'].get(clf_name, {}).get('epochs', 'N/A'),
    })

df_cost = pd.DataFrame(cost_rows)
print("=" * 90)
print("TABLE 9: Computational Cost Comparison")
print("=" * 90)
print(df_cost.to_string(index=False))
df_cost.to_csv(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'table9_computational_cost.csv'), index=False)


In [ ]:
# ==============================================================================
# TABLE 6 & Figure 7: Differentiation Matrix
# ==============================================================================
methods = ['AMEEGNet','EEG-DCNet','CIACNet','CLTNet','EEGEncoder',
           'MSCFormer','BrainGridNet','Multi-day','Feat.Rew.','LLM-EEG\n(Ours)']
features = ['Adaptive Preproc.','Decision Valid.','LLM Explain.',
            'Cross-trial Learn.','Plugin Arch.','Multi-classifier',
            'Signal Quality FB','Attention Mech.','End-to-end DL',
            'Spatial Features','Temporal Features','Cross-subject','Stat. Tests']
matrix = np.array([
    [0,0,0,0,0,0,0,0,0,1],[0,0,0,0,0,0,0,0,0,1],[0,0,0,0,0,0,0,0,0,1],
    [0,0,0,0,0,0,0,0,0,1],[0,0,0,0,0,0,0,0,0,1],[0,0,0,0,0,0,0,0,0,1],
    [0,0,0,0,0,0,0,0,0,1],[1,1,1,1,1,1,0,0,0,0],[1,1,1,1,1,1,1,1,1,1],
    [1,1,1,1,1,1,1,1,1,1],[1,1,1,1,1,1,1,1,1,1],[0,0,0,0,1,0,0,0,0,0],
    [1,1,1,1,1,1,1,1,1,1]])
fig, ax = plt.subplots(figsize=(14, 8))
cmap = sns.color_palette(['#F0F0F0', '#2CA02C'], as_cmap=True)
sns.heatmap(matrix, annot=True, fmt='d', cmap=cmap,
            xticklabels=methods, yticklabels=features,
            ax=ax, linewidths=0.5, linecolor='white')
ax.set_title('Figure 7: Feature Differentiation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results', 'fig7_diff_heatmap.png'))
plt.show()
print("\nKey: LLM-EEG is the ONLY framework with adaptive preprocessing + decision validation + explainability.")


## 9. Discussion & Conclusions <a name="sec9"></a>

### Key Findings
1. **APA** learns signal-quality-dependent preprocessing selection via Q-learning.
2. **DVA** increases reliability of accepted classifications.
3. **Inter-subject variability** is consistent with literature.

### Differentiation
| Aspect | Literature (2024-2025) | LLM-EEG |
|--------|----------------------|---------|
| Preprocessing | Static | Dynamic RL-based |
| Classification | Single model | Multi-model + DVA validation |
| Explainability | None | LLM-generated |
| Adaptation | Offline only | Online cross-trial Q-learning |
| Architecture | Monolithic | Modular plugin |

### Limitations
- Results shown use synthetic data when real BCI IV-2a files are not present; session-wise evaluation is enabled when A0xE.mat files are available.
- LLM integration requires API access.
- APA needs sufficient exploration.
- DVA thresholds may need per-subject tuning.

### Future Work
1. Real BCI IV-2a, IV-2b, HGD evaluation.
2. Phi-3 LLM integration.
3. DQN for continuous states.
4. Online BCI experiments.
5. Cross-subject transfer learning.

## 10. Export & References <a name="sec10"></a>

In [ ]:
# ==============================================================================
# Export All Results
# ==============================================================================
out = RESULTS_DIR if 'RESULTS_DIR' in dir() else 'results'
def conv(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.bool_): return bool(o)
    return o
with open(os.path.join(out, 'full_results.json'), 'w') as f:
    json.dump(results, f, indent=2, default=conv)
if 'df_summary' in dir():
    df_summary.to_csv(os.path.join(out, 'table1_summary.csv'), index=False)
if 'df_per_subj' in dir():
    df_per_subj.to_csv(os.path.join(out, 'table2_per_subject.csv'))
if 'df_lit' in dir():
    df_lit.to_csv(os.path.join(out, 'table5_literature.csv'), index=False)
print(f"\nResults exported to: {out}")
for fn in sorted(os.listdir(out)):
    sz = os.path.getsize(os.path.join(out, fn)) / 1024
    print(f"  {fn} ({sz:.1f} KB)")
print("\n" + "=" * 60)
print("EXPERIMENT COMPLETE!")
print("=" * 60)


## References

[1] Wolpaw et al., "Brain-computer interfaces," Clinical Neurophysiology, 2002.
[2] **AMEEGNet** (2025) -- Attention-based multiscale EEGNet. Acc: 81.17% (IV-2a), 89.83% (IV-2b), 95.49% (HGD).
[3] **EEG-DCNet** (2024) -- Dilated CNN. Acc: 83.31% (IV-2a).
[4] **Feature Reweighting** (2025) -- EEGNet feature reweighting.
[5] **CIACNet** (2025) -- CNN+CBAM+TCN. DOI: 10.3389/fnins.2025.1543508. Acc: 85.15%, kappa 0.80.
[6] **CLTNet** (2025) -- CNN+LSTM+Transformer. DOI: 10.3390/brainsci15020124. Acc: 83.02%, kappa 0.77.
[7] **Multi-day EEG** (2025) -- Novel dataset study. DOI: 10.1038/s41597-025-04826-y.
[8] **EEGEncoder** (2025) -- TCN+Transformer. DOI: 10.1038/s41598-025-06364-4. Acc: 86.46%.
[9] **MSCFormer** (2025) -- Multi-scale Conv+Transformer. DOI: 10.1038/s41598-025-96611-5. Acc: 82.95%.
[10] **BrainGridNet** (2025) -- Two-branch CNN. DOI: 10.1016/j.neunet.2023.11.037. Acc: 80.26%.
[11] Lawhern et al., "EEGNet," J. Neural Eng., 2018.
[12] Schirrmeister et al., "Deep learning with CNNs for EEG," Human Brain Mapping, 2017.
[13] Brunner et al., "BCI Competition 2008 -- Graz data set A." URL: https://www.bbci.de/competition/iv/
[14] Watkins & Dayan, "Q-learning," Machine Learning, 1992.

## Appendices

### Appendix A: Hyperparameters

| Component | Parameter | Value |
|-----------|-----------|-------|
| APA | alpha / gamma | 0.1 / 0.99 |
| APA | epsilon start/decay/min | 1.0 / 0.995 / 0.01 |
| APA | States / Actions | 64 / 3 |
| APA | Reward weights | SNR:0.25, Art:0.25, Disc:0.20, Acc:0.30 |
| DVA | Accept / Reject thr | 0.80 / 0.50 |
| DVA | Weights | Conf:0.4, Margin:0.3, Qual:0.3 |
| CSP | Components / Reg | 6 / 0.01 |
| EEGNet | F1/D/F2 | 8/2/16 |
| EEGNet | Epochs/LR/Batch | 100/0.001/32 |

### Appendix B: Preprocessing Profiles

| Profile | Bandpass (Hz) | Order | Notch Q | Artifact Thr (uV) |
|---------|:---:|:---:|:---:|:---:|
| Conservative | 4-40 | 4 | 35 | 150 |
| Moderate | 8-30 | 5 | 30 | 100 |
| Aggressive | 8-25 | 6 | 25 | 75 |